# RF Threat Classification System

## Building Environment

In [ ]:
!pip install -q nest-asyncio mcp langchain-openai langchain-mcp-adapters langgraph langchain-ollama gradio tabulate soundfile librosa seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q '/content/drive/MyDrive/ARL/processed_224.zip' -d './processed_data/'

!mkdir -p ARL
base="https://github.com/James-Tiny-Tjib/ARL/raw/main/ARL"
!curl -L -o "ARL/Airport_Noise_Dataset.zip" "$base/Airport%20Noise%20Dataset-20260603T141901Z-3-001.zip"
!curl -L -o "ARL/DroneAudioDataset.zip" "$base/DroneAudioDataset.zip"
!curl -L -o "ARL/best_cnn.pt" "$base/best_cnn.pt"
!curl -L -o "ARL/dca.png" "$base/dca.png"
!curl -L -o "ARL/dca_plain.png" "$base/dca_plain.png"
!curl -L -o "ARL/drone.png" "$base/drone.png"
!curl -L -o "ARL/drone_demo_dataset.zip" "$base/drone_demo_dataset.zip"
!curl -L -o "ARL/drone_multi_classifier.pt" "$base/drone_multi_classifier.pt"
!curl -L -o "ARL/fusion_model.pkl" "$base/fusion_model.pkl"
!curl -L -o "ARL/resnet50_drone_weights.pth" "$base/resnet50_drone_weights.pth"
!curl -L -o "ARL/sensor_client.pkl" "$base/sensor_client.pkl"
!curl -L -o "ARL/xgboostmodel.pkl" "$base/xgboostmodel.pkl"

!mkdir -p ARL/llama_drone_model_gguf
!curl -L -o "ARL/llama_drone_model_gguf/Modelfile" "$base/llama_drone_model_gguf/Modelfile"
!curl -L -o "ARL/llama_drone_model_gguf/llama-3.2-1b-instruct.Q4_K_M.gguf" "$base/llama_drone_model_gguf/llama-3.2-1b-instruct.Q4_K_M.gguf"


In [ ]:
import os, io, csv, time, random, threading, subprocess, asyncio
import base64, pickle, tempfile, shutil
from collections import Counter, defaultdict
from pathlib import Path
from itertools import cycle
import copy

import numpy as np
import torch
import torch as t
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset, ConcatDataset
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import r2_score, accuracy_score, classification_report
import librosa
import soundfile as sf
import pandas as pd

import gradio as gr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from io import BytesIO

import nest_asyncio
from mcp.server.fastmcp import FastMCP
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_ollama import ChatOllama

In [ ]:
BASE_DIR          = "./ARL"
AUDIO_MODEL_PATH  = f"{BASE_DIR}/drone_multi_classifier.pt"
RF_MODEL_PATH     = f"{BASE_DIR}/sensor_client.pkl"
VISUAL_MODEL_PATH = f"{BASE_DIR}/resnet50_drone_weights.pth"
MULTI_MODAL_MODEL_PATH = f"{BASE_DIR}/xgboostmodel.pkl"
SNAPSHOT_CSV_PATH = "./ARL/snapshot_log.csv"
DEMO_IMAGES_DIR   = "./drone_demo_dataset"
AUDIO_DATASET_DIR = "./DroneAudioDataset/Multiclass_Drone_Audio"
VISUAL_DATASET_DIR = "./drone_demo_dataset"

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLE_RATE = 4000
NUM_SENSORS = 23
MCP_PORT    = 8001

print(f"Device: {DEVICE}")

## Data Preparation

In [ ]:
# Drone audio dataset
ZIP_PATH = f"{BASE_DIR}/DroneAudioDataset.zip"
if os.path.exists(ZIP_PATH):
    print(f"Unzipping {ZIP_PATH}...")
    # FIX: Extract into "./DroneAudioDataset" instead of "."
    os.system(f'unzip -o "{ZIP_PATH}" -d "./DroneAudioDataset"')
elif not os.path.exists(AUDIO_DATASET_DIR):
    print("Zip not found, cloning from GitHub...")
    os.system("git clone https://github.com/saraalemadi/DroneAudioDataset.git")
else:
    print("DroneAudioDataset already exists, skipping.")

# Drone Demo Dataset
DEMO_PATH = f"{BASE_DIR}/drone_demo_dataset.zip"
if os.path.exists(DEMO_PATH):
    print(f"Unzipping {DEMO_PATH}...")
    os.system(f'unzip -o "{DEMO_PATH}" -d "./drone_demo_dataset"')
else:
    print(" already exists, skipping.")


# Background (airport) noise
BG_DIR      = f"{AUDIO_DATASET_DIR}/bg noise"
AIRPORT_ZIP = f"{BASE_DIR}/Airport_Noise_Dataset.zip"
os.makedirs(BG_DIR, exist_ok=True)
existing = len([f for f in os.listdir(BG_DIR) if f.endswith('.wav')]) if os.path.exists(BG_DIR) else 0
if existing > 0:
    print(f"bg noise already has {existing} .wav files, skipping.")
elif os.path.exists(AIRPORT_ZIP):
    tmp = tempfile.mkdtemp()
    os.system(f'unzip -o "{AIRPORT_ZIP}" -d "{tmp}"')
    count = 0
    for root, _, files in os.walk(tmp):
        for fn in files:
            if fn.endswith('.wav'):
                shutil.copy(os.path.join(root, fn), os.path.join(BG_DIR, fn))
                count += 1
    shutil.rmtree(tmp)
    print(f"Copied {count} .wav files into bg noise.")
else:
    print("No airport noise zip found.")

In [ ]:
# Dataset summary
if os.path.exists(AUDIO_DATASET_DIR):
    for item in sorted(os.listdir(AUDIO_DATASET_DIR)):
        full = os.path.join(AUDIO_DATASET_DIR, item)
        n = len([f for f in os.listdir(full) if f.endswith('.wav')]) if os.path.isdir(full) else 0
        print(f"  {item}/  ({n} .wav files)")

## Model Architectures

In [ ]:
# ── Audio: DroneCNN ──────────────────────────────────────────────────────────
class DroneCNN(nn.Module):
    """3-class mel-spectrogram classifier: Mambo / Bebop / Background."""
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(64 * 32 * 11, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 3),
        )

    def forward(self, x):
        return self.fc_layers(self.conv_layers(x))


# ── RF: IQCNN + Client ───────────────────────────────────────────────────────

class IQCNN(nn.Module):
    def __init__(self, num_classes=11):
        super(IQCNN, self).__init__()

        self.layer_dims=[]
        self.layers=nn.ModuleList()

        # Define convolutional layers here .......................................
        self.layers.append(nn.Conv1d(in_channels=2, out_channels=8, kernel_size=7, padding=3,bias=False))
        self.layers.append(nn.Conv1d(in_channels=8, out_channels=16, kernel_size=7, padding=3,bias=False))
        self.layers.append(nn.Conv1d(in_channels=16, out_channels=32, kernel_size=7, padding=3,bias=False))
        self.layers.append(nn.Conv1d(in_channels=32, out_channels=64, kernel_size=7, padding=3,bias=False))

        self.conv_num=len(self.layers)
        for i in range(self.conv_num):
          in_ch=self.layers[i].in_channels
          ker_sz=self.layers[i].kernel_size[0]
          self.layer_dims.append(in_ch*ker_sz)

        #Define linear layers here .....................................................
        self.layers.append(nn.Linear(64, 256,bias=False))
        self.layers.append(nn.Linear(256, num_classes,bias=False))

        for i in range(self.conv_num,len(self.layers)):
          self.layer_dims.append(self.layers[i].in_features)
        self.layer_dims.append(num_classes)
        # print(self.layer_dims)

        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)

        for i in range(self.conv_num):
          nn.init.xavier_uniform_(self.layers[i].weight)
        for i in range(self.conv_num,len(self.layers)-1):
            nn.init.kaiming_normal_(self.layers[i].weight, nonlinearity='relu')

        nn.init.kaiming_normal_(self.layers[i].weight, nonlinearity='linear')

    def forward(self, x):

        for i in range(self.conv_num):
          x=F.relu(self.layers[i](x))


        x = self.global_avg_pool(x)  # (batch_size, 64, 1)
        x = x.squeeze(-1)  # (batch_size, 64)

        for i in range(self.conv_num,len(self.layers)-1):
          x=F.relu(self.layers[i](x))
        x = self.layers[-1](x)  # (batch_size, output_shape)

        return x#F.log_softmax(x, dim=1)  # Use log_softmax for classification

def build_model(num_classes=2,model_name='IQCNN'):
  return IQCNN(num_classes=num_classes)#IQCNN(num_classes=11)

class Client:
  def __init__(self,dataset,num_classes=2,device='cpu'):
      self.device=device
      self.model = build_model(num_classes=num_classes).to(self.device)
      self.dataset =dataset
      self.dataloader = cycle(t.utils.data.DataLoader(self.dataset, batch_size=512, shuffle=True))
      self.cross_loss = nn.CrossEntropyLoss()
      self.optimizer = t.optim.Adam(self.model.parameters(), lr=0.001)

  def load_model(self,state_dict):
    self.model.load_state_dict(state_dict)

  def load_model_from_file(self,filename):
    state_dict=read_data(filename)
    self.load_model(state_dict)

  def save_model(self,filename):
    save_data(self.model.state_dict(),filename)

  def client_loss(self,pred,y):
    return self.cross_loss(pred,y)

  def evaluate(self,test_loader):
    correct,total=0,0
    with t.no_grad():
        for data in test_loader:
            x, y = data
            x=x.to(self.device)
            y=y.to(self.device)
            output = self.model(x)
            for idx, i in enumerate(output):
                if t.argmax(i) == y[idx]:
                    correct +=1
                total +=1
    print(f'accuracy: {round(correct/total, 3)}')
    return round(correct/total, 3)


  def train_batch(self):
    x,y = next(self.dataloader)
    x=x.to(self.device)
    y=y.to(self.device)
    self.optimizer.zero_grad()
    pred = self.model(x)
    loss=self.client_loss(pred,y)
    loss.backward()
    self.optimizer.step()
    return loss.item()

  def train(self,test_loader,epochs=int(1e3),echo=True,eval_acc=False):
    running_loss=0
    for epoch in range(epochs):
      epoch_loss = self.train_batch()
      running_loss+=epoch_loss

      if echo:
        if epoch%int(epochs/10) ==0 and epoch!=0:
          print("epoch:"+str(epoch)+" loss:"+ str(running_loss/int(epochs/10)))
          running_loss=0
          if eval_acc:
            self.evaluate(test_loader)
        # if epoch%(echo*10)==0 and epoch!=0:
        #   self.evaluate()

    return running_loss/int(epochs/10)


# ── Visual: ResNet50 ─────────────────────────────────────────────────────────
def build_visual_model(num_classes=1):
    """ResNet50 binary drone detector."""
    model = models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

## Data Loaders

In [ ]:
AUDIO_DATA_PATHS = {
    "mambo":      f"{AUDIO_DATASET_DIR}/membo_1",
    "bebop":      f"{AUDIO_DATASET_DIR}/bebop_1",
    "background": f"{AUDIO_DATASET_DIR}/bg noise",
}


class DroneAudioDataset(Dataset):
    """Loads .wav files and returns (mel-spectrogram tensor, label) pairs."""
    def __init__(self, files, labels, sr=22050, duration=1.0):
        self.files, self.labels, self.sr, self.duration = files, labels, sr, duration

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        y, sr = librosa.load(self.files[idx], duration=self.duration, sr=self.sr)
        n = int(self.sr * self.duration)
        if len(y) < n:
            y = np.pad(y, (0, n - len(y)))
        spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        spec_db = librosa.power_to_db(spec, ref=np.max)
        spec_db = (spec_db - spec_db.mean()) / (spec_db.std() + 1e-8)
        return torch.tensor(spec_db, dtype=torch.float32).unsqueeze(0), torch.tensor(self.labels[idx], dtype=torch.long)


def get_audio_loaders(data_paths=AUDIO_DATA_PATHS, max_per_class=800, batch_size=32, test_split=0.2):
    """Returns (train_loader, test_loader) for the drone audio dataset."""
    all_files, all_labels = [], []
    for label, (name, path) in enumerate(data_paths.items()):
        if not os.path.exists(path):
            print(f"Warning: {path} not found, skipping {name}")
            continue
        files = [os.path.join(path, f) for f in os.listdir(path) if f.endswith('.wav')][:max_per_class]
        all_files.extend(files)
        all_labels.extend([label] * len(files))
    X_tr, X_te, y_tr, y_te = train_test_split(all_files, all_labels, test_size=test_split)
    return (
        DataLoader(DroneAudioDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True),
        DataLoader(DroneAudioDataset(X_te, y_te), batch_size=batch_size),
    )


def create_synthetic_data(noise_std,grid_sz,grid_step,num_datapoints=int(1e3)):
  seq_sz=128
  adversary_pwr=1
  num_classes=2 # 0: good 1: adversary

  grid_dist=[i for i in range(int(grid_sz/grid_step))]

  # Scales for Unit Power
  scale_qpsk = 1.0 / np.sqrt(2)
  scale_16qam = 1.0 / np.sqrt(10)

  mod_options = ['BPSK', 'QPSK', '16-QAM']
  label_dict={'BPSK':0, 'QPSK':1, '16-QAM':2}
  data=[]
  labels=[]

  for c in range(num_classes):
    for d in grid_dist:
      for _ in range(num_datapoints):
        p=np.random.uniform(0, 1)
        rx_i, rx_q = np.array([]), np.array([])
        bpsk_sig=(2*np.random.randint(0,2,size=seq_sz)-1) + 0j
        qpsk_sig= (2*np.random.randint(0,2,size=seq_sz)-1) + 1j*(2*np.random.randint(0,2,seq_sz)-1)
        qpsk_sig=scale_qpsk*qpsk_sig

        merge_vec=np.random.uniform(0,1,size=seq_sz)
        merge_vec=merge_vec<=p

        sig=(merge_vec)*bpsk_sig+(1-merge_vec)*qpsk_sig

        if c==1:
          mp = np.array([-3, -1, 1, 3])
          adv_sig= mp[np.random.randint(0,4,size=seq_sz)] + 1j*mp[np.random.randint(0,4,size=seq_sz)]
          adv_sig=adv_sig*scale_16qam
          if d!=0:
            sig=(adversary_pwr/d)*adv_sig+sig
          else:
            sig=adv_sig+sig

        noise = noise_std * (np.random.randn(seq_sz) +1j * np.random.randn(seq_sz))
        sig=sig+noise

        data.append(np.array([np.real(sig),np.imag(sig)]))
        labels.append(c)

  X=np.array(data)
  Y=np.array(labels)
  print(X.shape,Y.shape)

  X_train_ar, X_test_ar, y_train_ar, y_test_ar = train_test_split(X, Y, test_size=0.3, random_state=42, stratify=Y) # split the data (70% train, 30% test)
  print(X_train_ar.shape, X_test_ar.shape, y_train_ar.shape, y_test_ar.shape)

  X_train = t.from_numpy(X_train_ar).float()
  y_train = t.from_numpy(y_train_ar).long() # Use long for integer labels
  X_test = t.from_numpy(X_test_ar).float()
  y_test = t.from_numpy(y_test_ar).long()
  trainset = TensorDataset(X_train, y_train)
  testset = TensorDataset(X_test, y_test)

  batch_size = 512

  train_loader = t.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)
  test_loader = t.utils.data.DataLoader(testset, batch_size=10, shuffle=True)

  return train_loader,test_loader,trainset,testset


def generate_sensor_rf_sample(noise_std,p,c,d):
  """
  p: bpsk qpsk prob
  c: 1 adversary
  d: distance
  """
  seq_sz=128
  adversary_pwr=1
  # Scales for Unit Power
  scale_qpsk = 1.0 / np.sqrt(2)
  scale_16qam = 1.0 / np.sqrt(10)

  bpsk_sig=(2*np.random.randint(0,2,size=seq_sz)-1) + 0j
  qpsk_sig= (2*np.random.randint(0,2,size=seq_sz)-1) + 1j*(2*np.random.randint(0,2,seq_sz)-1)
  qpsk_sig=scale_qpsk*qpsk_sig

  merge_vec=np.random.uniform(0,1,size=seq_sz)
  merge_vec=merge_vec<=p

  sig=(merge_vec)*bpsk_sig+(1-merge_vec)*qpsk_sig

  if c==1:
    mp = np.array([-3, -1, 1, 3])
    adv_sig= mp[np.random.randint(0,4,size=seq_sz)] + 1j*mp[np.random.randint(0,4,size=seq_sz)]
    adv_sig=adv_sig*scale_16qam
    if d!=0:
      sig=(adversary_pwr/d)*adv_sig+sig
    else:
      sig=adv_sig+sig

  noise = noise_std * (np.random.randn(seq_sz) +1j * np.random.randn(seq_sz))
  sig=sig+noise

  sample=t.from_numpy(np.array([np.real(sig),np.imag(sig)])).to(device)
  sample=sample.to(t.float32)
  return sample.unsqueeze(0)

visual_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


def get_visual_loaders(data_dir=VISUAL_DATASET_DIR, batch_size=64, test_split=0.2, val_split=0.1):
    """
    Returns (train_loader, val_loader, test_loader) for the drone visual dataset.
    Expects an ImageFolder layout:  data_dir/birds/  data_dir/drones/  data_dir/planes/
    Labels are remapped to binary:  drone=1, everything else=0.
    """
    from torchvision import datasets
    from torch.utils.data import random_split

    dataset = datasets.ImageFolder(root=data_dir, transform=visual_transforms)
    print(f"Visual dataset classes: {dataset.class_to_idx}")

    # Remap to binary:  drone → 1, bird/plane → 0
    new_samples, new_targets = [], []
    drone_idx = dataset.class_to_idx.get("drones", dataset.class_to_idx.get("drone", -1))
    for path, old_label in dataset.samples:
        binary = 1 if old_label == drone_idx else 0
        new_samples.append((path, binary))
        new_targets.append(binary)
    dataset.samples = new_samples
    dataset.targets = new_targets

    # Split
    total   = len(dataset)
    val_sz  = int(val_split * total)
    test_sz = int(test_split * total)
    train_sz = total - val_sz - test_sz
    print(f"Visual split -> Train: {train_sz} | Val: {val_sz} | Test: {test_sz}")

    train_ds, val_ds, test_ds = random_split(
        dataset, [train_sz, val_sz, test_sz],
        generator=torch.Generator().manual_seed(42),
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader

## Load Pretrained Models

In [ ]:
def save_data(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)

def read_data(path):
    class _CPU(pickle.Unpickler):
        def find_class(self, module, name):
            if module == 'torch.storage' and name == '_load_from_bytes':
                return lambda b: torch.load(io.BytesIO(b), map_location='cpu', weights_only=False)
            return super().find_class(module, name)
    with open(path, 'rb') as f:
        return _CPU(f).load()


def load_audio_model(path=AUDIO_MODEL_PATH):
    model = DroneCNN().to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    print(f"Loaded audio model from {path}")
    return model


def load_rf_model(path=RF_MODEL_PATH):
    rf_weights = read_data(path)
    model = build_model(num_classes=2)
    model.load_state_dict(rf_weights)
    print(f"Loaded RF model from {path}")
    return model


def load_visual_model(path=VISUAL_MODEL_PATH):
    model = build_visual_model().to(DEVICE)
    checkpoint = torch.load(path, map_location=DEVICE)
    model.load_state_dict(checkpoint.get('model_state_dict', checkpoint))
    model.eval()
    print(f"Loaded visual model from {path}")
    return model

def load_multi_modal_model(path = MULTI_MODAL_MODEL_PATH):
    model = read_data(path)
    print(f"Loaded Multi_Modal from {path}")
    return model


AU_MODEL  = load_audio_model()
RF_MODEL    = load_rf_model()
VS_MODEL = load_visual_model()
FS_MODEL = load_multi_modal_model()

## Training

In [ ]:
def train_audio_model(save_path=AUDIO_MODEL_PATH, epochs=10, batch_size=32):
    """Train DroneCNN on drone audio dataset and save weights."""
    train_loader, test_loader = get_audio_loaders(batch_size=batch_size)
    model = DroneCNN().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(x), y).backward()
            optimizer.step()
        print(f"Epoch {epoch+1}/{epochs}")
    # Evaluate
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            correct += (model(x).argmax(1) == y).sum().item()
            total += y.size(0)
    print(f"Test accuracy: {correct/total:.3f}")
    torch.save(model.state_dict(), save_path)
    print(f"Saved -> {save_path}")
    return model


def train_rf_client(save_path=RF_MODEL_PATH, epochs=2000):
    model = None

    # Free GPU memory from other models before training
    train_loader,test_loader,trainset,testset=create_synthetic_data(noise_std=0.1,grid_sz=80,grid_step=4,num_datapoints=int(1e4))
    device= 'cuda' if t.cuda.is_available() else 'cpu'
    print(device)

    rf_client= Client(trainset,device=device)
    rf_client.train(test_loader,epochs=epochs,echo=True,eval_acc=True)
    rf_client.evaluate(test_loader)
    rf_client.save_model(RF_MODEL_PATH)
    return rf_client

def train_visual_model(save_path=VISUAL_MODEL_PATH, epochs=5, batch_size=64, lr=0.001):
    """
    Train ResNet50 (transfer-learned) binary drone detector and save weights as .pth.
    Freezes the backbone and only trains the final FC layer, matching the
    original Colab training procedure.
    """
    train_loader, val_loader, test_loader = get_visual_loaders(batch_size=batch_size)

    # Build model with pretrained ImageNet backbone
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    for param in model.parameters():
        param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, 1)
    model = model.to(DEVICE)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=lr)

    best_val_acc = 0.0

    for epoch in range(epochs):
        # ── Training ──
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for inputs, labels in train_loader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE).float().view(-1, 1)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc  = correct / total

        # ── Validation ──
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(DEVICE)
                labels = labels.to(DEVICE).float().view(-1, 1)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                preds = (torch.sigmoid(outputs) >= 0.5).float()
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total
        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss/val_total:.4f}  Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc

    # ── Test ──
    model.eval()
    test_correct, test_total = 0, 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE).float().view(-1, 1)
            preds = (torch.sigmoid(model(inputs)) >= 0.5).float()
            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)
    print(f"Test Accuracy: {test_correct/test_total:.4f}")

    # Save as .pth state_dict (compatible with load_visual_model)
    torch.save(model.state_dict(), save_path)
    print(f"Saved visual model -> {save_path}")
    return model

# Uncomment to retrain from scratch:
# train_audio_model()
# train_rf_client()
# train_visual_model()

## Interface

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from pathlib import Path
import gc
import os
import io
import time
import threading
import subprocess
import asyncio
import librosa
from collections import Counter
import tempfile
import soundfile as sf
import copy

import gradio as gr
import numpy as np
import matplotlib.pyplot as plt
import nest_asyncio
from io import BytesIO
from PIL import Image
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_ollama import ChatOllama

import base64
import random
import csv
from tabulate import tabulate

import torch as t  # alias used by generate_sensor_rf_sample
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()

univ_hop_dur = 0.8
univ_sph = 8


IMAGE_PATH = "./ARL/dca.png"
DRONE_IMAGE_PATH = "./ARL/drone.png"
IS_DRONE_IN_RANGE = False
IS_DRONE_AUDIO_RANGE = False
IMAGE_WIDTH_PX = 500
IMAGE_HEIGHT_PX = 693

SENSOR_RADIUS_PX = 50
DRONE_RADIUS_PX = 10  # since the drone is 20x20

def load_drone_as_data_uri(path, white_thresh=200):
    if not os.path.exists(path):
        return ""

    img = Image.open(path).convert("RGBA")
    data = img.getdata()

    new_data = []
    for r, g, b, a in data:
        if r >= white_thresh and g >= white_thresh and b >= white_thresh:
            new_data.append((255, 255, 255, 0))
        else:
            new_data.append((r, g, b, a))

    img.putdata(new_data)

    buffer = BytesIO()
    img.save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{encoded}"

# Drone Movement Parameters
CURRENT_POS = (
    random.randint(DRONE_RADIUS_PX, IMAGE_WIDTH_PX - DRONE_RADIUS_PX),
    random.randint(DRONE_RADIUS_PX, IMAGE_HEIGHT_PX - DRONE_RADIUS_PX),
)
LAST_MOVE_TIME = None

SENSORS = [
    (235, 37), (190, 89), (167, 135), (190, 221), (165, 287),
    (179, 350), (164, 417), (106, 462), (105, 550), (161, 593),
    (226, 633), (307, 650), (383, 617), (403, 551), (420, 480),
    (431, 421), (435, 345), (429, 285), (412, 224), (394, 177),
    (373, 128), (332, 86), (305, 38)
]

NUM_SENSORS = len(SENSORS)  # must match SENSORS list (23 entries)

RF_VISIBILITY_RNG=80
GRID_STEP=4
SENSOR_PROBS=np.random.uniform(0,1,size=len(SENSORS))
REFRESH_RATE = 2.0

RF_CLIENT = None
RF_MODEL = None
AU_MODEL = None
VS_MODEL = None
FS_MODEL = None

def load_models():
    global RF_CLIENT, RF_MODEL, AU_MODEL, VS_MODEL, FS_MODEL
    RF_MODEL = load_rf_model()
    AU_MODEL  = load_audio_model()
    VS_MODEL = load_visual_model()
    FS_MODEL = load_multi_modal_model()

    RF_MODEL.to(device)
    AU_MODEL.to(device)
    VS_MODEL.to(device)

load_models()

# --- 1. CONFIGURATION ---
class SystemState:
    def __init__(self):
        self.fs = 4000
        self.chunk_duration = 4.0
        self.running = False
        self.new_data_event = threading.Event()
        self.current_waveform = np.array([])
        self.gt_iq = []
        self.gt_mod = []
        self.gt_fc = []
        self.demod = []
        self.cur_analysis = None
        self.cur_spec_base64 = None
        self.recent_audio_html = None
        self.last_sensor_states = []
        self.snapshot_count = 0
        self._last_is_threat = None
        self.alerts = []
        self.last_alert_time = 0
        self.smoothed_pos = None
        self.movement = None
        self.last_rendered_movement_id = None # Added for CSS guard
        self.confidence = 0.0
        self.is_threat  = False

        self.snapshot = {
            "timestamp": None,
            "drone_pos": None,
            "is_threat_gt": None,
            "sensor_states": None,
            "nearest_sensor": None,
            "nearest_sensor_dist": None,
            "rf": {
                "triggered_sensors": None,
                "confidence": None,
                "is_threat": None,
                "spectrogram_b64": None,
            },
            "audio": {
                "drone_in_range": None,
                "confidence": None,
                "is_threat": None,
                "spectrogram_b64": None,
                "audio_b64": None,
            },
            "visual": {
                "drone_in_range": None,
                "confidence": None,
                "is_threat": None,
                "classification": None,
                "image_b64": None,
            },
        }

        ### MASTER SENSOR LIST ###
        sensor_template = {
            "x" : None,
            "y" : None,
            "triggered" : None,
            "audio_triggered" : None,
            "rf_logits" : None,
            "au_logits" : None,
            "vs_logits" : None
        }

        self.master_snapshot = {
            "timestamp": None,
            "drone_pos": None,
            "is_threat_gt": None,
            "nearest_sensor": None,
            "nearest_sensor_dist": None,
            "fusion_confidence" : None,
            "sensor_list" : [sensor_template.copy() for _ in range(23)]
        }

        self.last_master_snapshot = None

        self.profiles = {
            "S1 (Long Range)": {"freqs": [10, 20], "mod": "BPSK", "color": "#00CCFF", "desc": "Robust / Low Rate"},
            "S2 (Standard)": {"freqs": [30, 40], "mod": "QPSK", "color": "#00FF00", "desc": "Standard Link"},
            "S3 (THREAT)": {"freqs": [50, 60, 70], "mod": "16-QAM", "color": "#FF0000", "desc": "High Speed Burst"}
        }

sys_state = SystemState()

def save_snapshot_to_csv(snapshot, path="./ARL/snapshot_log.csv"):
    sensor_states = snapshot["sensor_states"] or []
    sensor_row = {}
    for i, s in enumerate(sensor_states):
        sensor_row[f"sensor_{i}_threat_conf"]   = s.get("threat_conf", 0.0)
        sensor_row[f"sensor_{i}_friendly_conf"] = s.get("friendly_conf", 0.0)
    for i in range(len(sensor_states), len(SENSORS)):
        sensor_row[f"sensor_{i}_threat_conf"]   = 0.0
        sensor_row[f"sensor_{i}_friendly_conf"] = 0.0

    row = {
        "timestamp": snapshot["timestamp"],
        "is_threat_gt": snapshot.get("is_threat_gt", None),
        "rf_confidence": snapshot["rf"]["confidence"],
        "rf_is_threat": snapshot["rf"]["is_threat"],
        "audio_mambo": snapshot["audio"]["confidence"]["mambo"] if snapshot["audio"]["confidence"] else None,
        "audio_bebop": snapshot["audio"]["confidence"]["bebop"] if snapshot["audio"]["confidence"] else None,
        "audio_background": snapshot["audio"]["confidence"]["background"] if snapshot["audio"]["confidence"] else None,
        "audio_is_threat": snapshot["audio"]["is_threat"],
        "visual_confidence": snapshot["visual"]["confidence"],
        "visual_is_threat": snapshot["visual"]["is_threat"],
        **sensor_row
    }
    file_exists = os.path.exists(path)
    with open(path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)

def set_snapshot_default(state):
    dists = [np.sqrt((CURRENT_POS[0] - s[0])**2 + (CURRENT_POS[1] - s[1])**2) for s in SENSORS]
    nearest_idx = int(np.argmin(dists))
    nearest_dist = float(dists[nearest_idx])

    state.snapshot["timestamp"] = time.time()
    state.snapshot["drone_pos"] = CURRENT_POS
    state.snapshot["sensor_states"] = state.last_sensor_states
    state.snapshot["nearest_sensor"] = nearest_idx
    state.snapshot["nearest_sensor_dist"] = nearest_dist
    state.snapshot["is_threat_gt"] = int(getattr(state, "_last_is_threat", 0))
    state.snapshot["rf"]["triggered_sensors"] = state.cur_analysis
    state.snapshot["rf"]["is_threat"] = IS_DRONE_IN_RANGE
    state.snapshot["audio"]["drone_in_range"] = IS_DRONE_AUDIO_RANGE
    state.snapshot["visual"]["drone_in_range"] = IS_DRONE_IN_RANGE

def set_master_snapshot_default(state):
    dists = [np.sqrt((CURRENT_POS[0] - s[0])**2 + (CURRENT_POS[1] - s[1])**2) for s in SENSORS]
    nearest_idx = int(np.argmin(dists))
    nearest_dist = float(dists[nearest_idx])

    state.last_master_snapshot = state.master_snapshot

    state.master_snapshot["timestamp"] = time.time()
    state.master_snapshot["drone_pos"] = CURRENT_POS
    state.master_snapshot["nearest_sensor"] = nearest_idx
    state.master_snapshot["nearest_sensor_dist"] = nearest_dist
    state.master_snapshot["fusion_confidence"] = [0,0]

    state.master_snapshot["is_threat_gt"] = int(getattr(state, "_last_is_threat", 0))

# --- 4. STATUS CIRCLE + VIS PANEL ---
def point_to_segment_distance(px, py, x1, y1, x2, y2):
        dx = x2 - x1
        dy = y2 - y1

        if dx == 0 and dy == 0:
            return ((px - x1) ** 2 + (py - y1) ** 2) ** 0.5

        t = ((px - x1) * dx + (py - y1) * dy) / (dx * dx + dy * dy)
        t = max(0.0, min(1.0, t))

        closest_x = x1 + t * dx
        closest_y = y1 + t * dy

        return ((px - closest_x) ** 2 + (py - closest_y) ** 2) ** 0.5

# --- RANDOM PATH GENERATION IN PIXELS ---
def rand_pos():
    step = 40
    new_x = CURRENT_POS[0] + random.randint(-step, step)
    new_y = CURRENT_POS[1] + random.randint(-step, step)

    new_x = max(DRONE_RADIUS_PX, min(new_x, IMAGE_WIDTH_PX - DRONE_RADIUS_PX))
    new_y = max(DRONE_RADIUS_PX, min(new_y, IMAGE_HEIGHT_PX - DRONE_RADIUS_PX))
    return (new_x, new_y)

def current_threat_active():
    if not sys_state.gt_mod:
        return bool(np.random.binomial(n=1, p=0.8, size=1)[0])
    return "16-QAM" in sys_state.gt_mod

def drone_movement_and_detection():
    global CURRENT_POS, LAST_MOVE_TIME
    global IS_DRONE_IN_RANGE, IS_DRONE_AUDIO_RANGE

    now = time.time()
    dt = REFRESH_RATE if LAST_MOVE_TIME is None else now - LAST_MOVE_TIME
    LAST_MOVE_TIME = now

    p1 = CURRENT_POS
    p2 = rand_pos()
    CURRENT_POS = p2

    sys_state.movement = {
        "id": random.random(),
        "p1": p1,
        "p2": p2,
        "dt": dt
    }

    is_threat = current_threat_active()
    sensor_states = []
    any_triggered = False
    audio_triggered = False
    sys_state.cur_analysis = []

    for idx, sp in enumerate(SENSORS):
        sx, sy = sp
        dist = point_to_segment_distance(sx, sy, p1[0], p1[1], p2[0], p2[1])
        triggered = dist <= RF_VISIBILITY_RNG
        a_triggered = dist <= SENSOR_RADIUS_PX

        threat_conf = 0.0
        friendly_conf = 0.0

        if triggered:
            d = int(dist/GRID_STEP)
            sample = generate_sensor_rf_sample(0.1, SENSOR_PROBS[idx], is_threat, d)
            logits = RF_MODEL(sample)
            probs = torch.softmax(logits, dim=1)
            friendly_conf = probs[0][0].item()
            threat_conf = probs[0][1].item()
            if threat_conf >= 0.5:
                sys_state.cur_analysis.append(idx)

        # MS, Populate rf_logits
        d = int(dist/GRID_STEP)
        sample = generate_sensor_rf_sample(0.1, SENSOR_PROBS[idx], is_threat, d)
        logits = RF_MODEL(sample)
        any_triggered = any_triggered or triggered
        audio_triggered = audio_triggered or a_triggered

        sensor_states.append({
            "x": sx,
            "y": sy,
            "triggered": triggered,
            "audio_triggered": a_triggered,
            "threat_conf": threat_conf,
            "friendly_conf": friendly_conf,
        })

        # MS
        sensor_dict = sys_state.master_snapshot["sensor_list"][idx]
        sensor_dict["x"] = sx
        sensor_dict["y"] = sy
        sensor_dict["triggered"] = triggered
        sensor_dict["audio_triggered"] = a_triggered
        sensor_dict["rf_logits"] = logits[0].tolist()

    IS_DRONE_IN_RANGE = any_triggered
    IS_DRONE_AUDIO_RANGE = audio_triggered

    sys_state.last_sensor_states = sensor_states
    sys_state._last_is_threat = is_threat
    sys_state.snapshot["is_threat_gt"] = any_triggered
    sys_state.master_snapshot["is_threat_gt"] = any_triggered


def build_status_circle_html():
    movement = getattr(sys_state, 'movement', None)

    if movement is None:
        p1 = p2 = CURRENT_POS
        movement_duration = 0.0
    else:
        # THE CSS RE-RENDER GUARD
        # Only start a new animation if the movement ID is genuinely fresh
        if movement["id"] != getattr(sys_state, "last_rendered_movement_id", None):
            p1 = movement["p1"]
            p2 = movement["p2"]
            sys_state.last_rendered_movement_id = movement["id"]
            movement_duration = max(0.1, movement["dt"])
        else:
            # We already rendered this move. If Gradio refreshes, lock the drone at its destination.
            p1 = movement["p2"]
            p2 = movement["p2"]
            movement_duration = 0.0

    is_threat = getattr(sys_state, '_last_is_threat', False)
    sensor_states = getattr(sys_state, 'last_sensor_states', [])

    animation_name = f"moveDot{random.randint(0, 100000)}"

    img_src = ""
    if os.path.exists(IMAGE_PATH):
        with open(IMAGE_PATH, "rb") as f:
            encoded = base64.b64encode(f.read()).decode("utf-8")
        ext = IMAGE_PATH.split(".")[-1].lower()
        if ext == "jpg":
            ext = "jpeg"
        img_src = f"data:image/{ext};base64,{encoded}"

    drone_img_src = load_drone_as_data_uri(DRONE_IMAGE_PATH)

    drone_filter = (
      "brightness(0) invert(0.6)" if not is_threat else
      "brightness(0) saturate(100%) invert(20%) sepia(100%) saturate(6000%) hue-rotate(0deg) brightness(1.2)"
    )

    sensor_html = []
    for idx, sensor in enumerate(sensor_states):
        sx = sensor["x"]
        sy = sensor["y"]
        triggered = sensor["triggered"]

        sensor_color = "rgba(255, 165, 0, 0.16)" if triggered else "rgba(100, 180, 255, 0.10)"
        sensor_border = "rgba(255, 180, 80, 0.40)" if triggered else "rgba(120, 190, 255, 0.22)"

        sensor_html.append(f"""
            <div style="position: absolute; left: {sx}px; top: {sy}px; width: {2 * SENSOR_RADIUS_PX}px; height: {2 * SENSOR_RADIUS_PX}px; transform: translate(-50%, -50%); border-radius: 50%; background: {sensor_color}; border: 1.5px solid {sensor_border}; box-shadow: 0 0 12px {sensor_color}; pointer-events: none; display: flex; align-items: center; justify-content: center;"><span style="color: rgb(140, 195, 255); font-size: 14px; font-weight: 700; font-family: monospace; opacity: 0.95; text-shadow: 0 0 4px rgba(0,0,0,0.7);">{idx}</span></div>
        """)

    sensors_html = "\n".join(sensor_html)

    return f"""
    <style>
    @keyframes {animation_name} {{
        0%   {{ left: {p1[0]}px; top: {p1[1]}px; }}
        100% {{ left: {p2[0]}px; top: {p2[1]}px; }}
    }}
    </style>

    <div style="height: 100%; min-height: {IMAGE_HEIGHT_PX + 32}px; display: flex; align-items: center; justify-content: center; background: #0f1117; border-radius: 18px; border: 1px solid #2a2f3a; padding: 16px; box-sizing: border-box;">
        <div style="position: relative; width: {IMAGE_WIDTH_PX}px; height: {IMAGE_HEIGHT_PX}px; min-width: {IMAGE_WIDTH_PX}px; max-width: {IMAGE_WIDTH_PX}px; min-height: {IMAGE_HEIGHT_PX}px; max-height: {IMAGE_HEIGHT_PX}px; overflow: hidden; border-radius: 18px; flex-shrink: 0;">
            <img src="{img_src}" style="width: {IMAGE_WIDTH_PX}px; height: {IMAGE_HEIGHT_PX}px; min-width: {IMAGE_WIDTH_PX}px; max-width: {IMAGE_WIDTH_PX}px; min-height: {IMAGE_HEIGHT_PX}px; max-height: {IMAGE_HEIGHT_PX}px; object-fit: fill; display: block; border-radius: 18px;">
            {sensors_html}
            <img src="{drone_img_src}" style="position: absolute; left: {p1[0]}px; top: {p1[1]}px; width: {2 * DRONE_RADIUS_PX * 2}px; height: {2 * DRONE_RADIUS_PX * 2}px; transform: translate(-50%, -50%); animation: {animation_name} {movement_duration}s linear forwards; filter: {drone_filter}; opacity: 0.95; pointer-events: none;">
        </div>
    </div>
    """

def build_blank_visual_panel():
    return """
    <div style="width: 100%; aspect-ratio: 1 / 1; min-height: 320px; background: #0f1117; border: 2px dashed #3b4252; border-radius: 18px; display: flex; align-items: center; justify-content: center; color: #8b949e; font-size: 20px; font-weight: 600;">
        Visualization Output
    </div>
    """

# --- RF SIGNAL GENERATOR (for spectrogram visualization) ---
def generate_mixed_chunk(noise_level=0.1):
    fs = sys_state.fs
    total_dur = sys_state.chunk_duration

    full_wave = []
    gt_iq = []
    gt_mod = []
    gt_fc = []
    demod = []
    current_t = 0

    while current_t < total_dur:
        hop_dur = univ_hop_dur
        seg_dur = np.random.randint(0, 5) * hop_dur + 1.0
        if current_t + seg_dur > total_dur:
            seg_dur = total_dur - current_t

        p_name = np.random.choice(list(sys_state.profiles.keys()))
        profile = sys_state.profiles[p_name]
        freq_list = profile["freqs"]
        mod_type = profile["mod"]

        num_hops = int(np.ceil(seg_dur / hop_dur))
        segment_wave = []

        for i in range(num_hops):
            this_hop_dur = hop_dur
            if this_hop_dur <= 0:
                break
            f_c = freq_list[i % len(freq_list)]
            t = np.linspace(0, this_hop_dur, int(fs * this_hop_dur), endpoint=False)
            num_syms = univ_sph

            if mod_type == "16-QAM":
                scale = 1.0 / np.sqrt(10)
                levs = [-3, -1, 1, 3]
                syms = [complex(x, y) * scale for x, y in zip(
                    np.random.choice(levs, num_syms), np.random.choice(levs, num_syms))]
            elif mod_type == "QPSK":
                scale = 1.0 / np.sqrt(2)
                syms = [(np.random.choice([-1, 1]) + 1j * np.random.choice([-1, 1])) * scale
                        for _ in range(num_syms)]
            else:  # BPSK
                syms = [(2 * np.random.randint(0, 2) - 1) + 0j for _ in range(num_syms)]

            gt_iq += syms
            gt_fc += [f_c] * len(syms)
            gt_mod += [mod_type] * len(syms)

            s_up = np.repeat(syms, int(len(t) / num_syms) + 1)[:len(t)]
            wave = s_up.real * np.cos(2 * np.pi * f_c * t) - s_up.imag * np.sin(2 * np.pi * f_c * t)

            demod_i = np.mean(np.array_split(wave * 2 * np.cos(2 * np.pi * f_c * t), num_syms), axis=1)
            demod_q = np.mean(np.array_split(-wave * 2 * np.sin(2 * np.pi * f_c * t), num_syms), axis=1)
            demod.append(demod_i + 1j * demod_q)
            segment_wave.append(wave)

        if segment_wave:
            full_wave.append(np.concatenate(segment_wave))
        current_t += seg_dur

    raw = np.concatenate(full_wave)
    raw = raw[:int(fs * total_dur)]
    sys_state.current_waveform = raw + (np.random.randn(len(raw)) * noise_level)
    sys_state.gt_iq = gt_iq
    sys_state.gt_fc = gt_fc
    sys_state.gt_mod = gt_mod

def generate_rf_spectrogram_base64():
    wave = sys_state.current_waveform
    fs = sys_state.fs
    if len(wave) == 0:
        return None

    plt.style.use('dark_background')
    fig, (ax_time, ax_spec) = plt.subplots(2, 1, figsize=(10, 6), facecolor='#0f1117')
    fig.subplots_adjust(hspace=0.45)

    t_axis = np.linspace(0, len(wave) / fs, len(wave))
    ax_time.plot(t_axis, wave, color='#00FF00', lw=0.6)
    ax_time.set_title("Time Domain", color='white', fontsize=11, pad=6)
    ax_time.set_xlabel("Time (s)", color='#aaaaaa', fontsize=9)
    ax_time.set_ylabel("Amplitude", color='#aaaaaa', fontsize=9)
    ax_time.set_xlim(0, t_axis[-1])
    ax_time.tick_params(colors='white')
    ax_time.set_facecolor('#0f1117')
    for spine in ax_time.spines.values():
        spine.set_edgecolor('#333333')

    nfft = min(512, len(wave) // 4)
    ax_spec.specgram(wave, NFFT=nfft, Fs=fs, noverlap=nfft // 2, cmap='inferno')
    ax_spec.set_title("Spectrogram  (Frequency vs Time)", color='white', fontsize=11, pad=6)
    ax_spec.set_xlabel("Time (s)", color='#aaaaaa', fontsize=9)
    ax_spec.set_ylabel("Frequency (Hz)", color='#aaaaaa', fontsize=9)
    ax_spec.tick_params(colors='white')
    ax_spec.set_facecolor('#0f1117')
    for spine in ax_spec.spines.values():
        spine.set_edgecolor('#333333')

    buf = BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', facecolor='#0f1117', dpi=120)
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode('utf-8')

def generate_audio_spectrogram_base64(audio, sr):
    plt.style.use('dark_background')
    fig, ax = plt.subplots(figsize=(10, 3.5), facecolor='#0f1117')

    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128, fmax=sr // 2)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel',
                                   ax=ax, cmap='inferno', fmax=sr // 2)
    fig.colorbar(img, ax=ax, format='%+2.0f dB')
    ax.set_title("Mel Spectrogram (Freq vs Time)", color='white', fontsize=11, pad=6)
    ax.set_xlabel("Time (s)", color='#aaaaaa', fontsize=9)
    ax.set_ylabel("Frequency (Hz)", color='#aaaaaa', fontsize=9)
    ax.tick_params(colors='white')
    ax.set_facecolor('#0f1117')
    for spine in ax.spines.values():
        spine.set_edgecolor('#333333')

    buf = BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', facecolor='#0f1117', dpi=120)
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode('utf-8')


def run_rf_detection():
    drone_movement_and_detection()
    set_snapshot_default(sys_state)
    set_master_snapshot_default(sys_state)
    snap = sys_state.snapshot["rf"]

    generate_mixed_chunk(0.1)
    sys_state.cur_spec_base64 = generate_rf_spectrogram_base64()
    snap["spectrogram_b64"] = sys_state.cur_spec_base64

    mods = list(set(sys_state.gt_mod)) if sys_state.gt_mod else []
    freqs = list(set(sys_state.gt_fc)) if sys_state.gt_fc else []

    if snap["triggered_sensors"] and isinstance(snap["triggered_sensors"], list):
        snap["confidence"] = len(snap["triggered_sensors"]) / len(SENSORS) * 100
        snap["is_threat"] = True
        return (
            f"RF spectrogram generated and displayed. "
            f"Anomalous RF transmissions detected at sensor indices: {snap['triggered_sensors']}. "
            f"Confidence: {snap['confidence']:.1f}%. "
            f"Modulation types in environment: {mods}. "
            f"Carrier frequencies observed: {freqs} Hz."
        )

    snap["confidence"] = 0.0
    snap["is_threat"] = False
    return (
        f"RF spectrogram generated and displayed. "
        f"No anomalous RF activity detected. "
        f"Modulation types in environment: {mods}. "
        f"Carrier frequencies observed: {freqs} Hz."
    )

def get_batched_audio_prediction_confidence(batched_tensor):
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    batched_tensor = batched_tensor.to(DEVICE)
    with torch.no_grad():
        output = AU_MODEL(batched_tensor)
        probs = torch.nn.functional.softmax(output, dim=1)

    results = []
    for i in range(probs.size(0)):
        results.append({
            "mambo": probs[i][0].item() * 100,
            "bebop": probs[i][1].item() * 100,
            "background": probs[i][2].item() * 100,
        })
    return results, output

def run_audio_detection():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    SR = 22050
    AIRPORT_DIR = "./DroneAudioDataset/Multiclass_Drone_Audio/bg noise"
    DRONE_DIRS = [
        "./DroneAudioDataset/Multiclass_Drone_Audio/membo_1",
        "./DroneAudioDataset/Multiclass_Drone_Audio/bebop_1"
    ]

    IS_DRONE_RANGE = sys_state.snapshot["audio"]["drone_in_range"]
    nearest = sys_state.snapshot["nearest_sensor"]
    dist = sys_state.snapshot["nearest_sensor_dist"]

    _drone_files = []
    for d in DRONE_DIRS:
        if os.path.exists(d):
            _drone_files.extend([os.path.join(d, f) for f in os.listdir(d) if f.endswith('.wav')])
    all_bg_files = [os.path.join(AIRPORT_DIR, f) for f in os.listdir(AIRPORT_DIR) if f.endswith('.wav')]

    spec_db_list = []
    nearest_attenuated_y = None
    TARGET_LEN = SR

    for idx, sensor_dict in enumerate(sys_state.master_snapshot["sensor_list"]):

        if sensor_dict["audio_triggered"]:
            source_file = random.choice(_drone_files)
            y, _ = librosa.load(source_file, sr=SR, duration=1.0)
            attenuated_y = y * (1.0 / (dist + 1e-6))
        else:
            source_file = random.choice(all_bg_files)
            attenuated_y, _ = librosa.load(source_file, sr=SR, duration=1.0)

        if len(attenuated_y) < TARGET_LEN:
            attenuated_y = np.pad(attenuated_y, (0, int(TARGET_LEN - len(attenuated_y))))
        elif len(attenuated_y) > TARGET_LEN:
            attenuated_y = attenuated_y[:TARGET_LEN]

        if idx == nearest:
            nearest_attenuated_y = attenuated_y

        spec = librosa.feature.melspectrogram(y=attenuated_y, sr=SR, n_mels=128)
        spec_db = librosa.power_to_db(spec, ref=np.max)
        spec_db = (spec_db - np.mean(spec_db)) / (np.std(spec_db) + 1e-6)

        spec_db_list.append(spec_db)

    batched_array = np.stack(spec_db_list)
    batch_tensor = torch.tensor(batched_array, dtype=torch.float32).unsqueeze(1)
    batch_confidences, logits = get_batched_audio_prediction_confidence(batch_tensor)

    for idx, sensor_dict in enumerate(sys_state.master_snapshot["sensor_list"]):
        sensor_dict["au_logits"] = [logits[idx][0].item(), logits[idx][1].item(), logits[idx][2].item()]

    conf = batch_confidences[nearest]
    drone_conf = conf["mambo"] + conf["bebop"]
    sys_state.snapshot["audio"]["confidence"] = conf
    sys_state.snapshot["audio"]["is_threat"] = drone_conf > 70
    buf = io.BytesIO()
    sf.write(buf, nearest_attenuated_y, SR, format='WAV')
    b64_audio = base64.b64encode(buf.getvalue()).decode("utf-8")
    audio_html = f"""
        <div style="margin-bottom: 10px; padding: 8px; background: #1e222c; border-radius: 8px; border-left: 4px solid #00CCFF;">
            <p style="margin: 0 0 5px 0; font-size: 12px; color: #8b949e;">Sensor {nearest} (Dist: {dist:.1f}px | Conf: {drone_conf:.1f}%)</p>
            <audio controls style="height: 30px; width: 100%;">
                <source src="data:audio/wav;base64,{b64_audio}" type="audio/wav">
            </audio>
        </div>
    """
    spec_b64 = generate_audio_spectrogram_base64(nearest_attenuated_y, SR)
    sys_state.snapshot["audio"]["spectrogram_b64"] = spec_b64
    sys_state.snapshot["audio"]["audio_b64"] = b64_audio
    if spec_b64:
        sys_state.recent_audio_html = f"""
            <div style="margin-bottom: 10px; padding: 8px; background: #1e222c; border-radius: 8px; border-left: 4px solid #00CCFF;">
                <p style="margin: 0 0 5px 0; font-size: 12px; color: #8b949e; font-weight: bold;">Mel Spectrogram (Freq vs Time)</p>
                <img src="data:image/png;base64,{spec_b64}" style="width: 100%; border-radius: 6px;">
            </div>
        """ + audio_html
    else:
        sys_state.recent_audio_html = audio_html
    if drone_conf > 70:
        return f"Acoustic analysis complete. Drone detected at closest sensor {nearest}. Confidence: {drone_conf:.1f}%. Mel spectrogram displayed."
    return "Acoustic analysis complete. Drone not detected. Mel spectrogram displayed."

def run_visual_detection():
    if VS_MODEL is None:
        return "Visual model not loaded yet."
    global global_visual_html

    demo_dir = Path("./drone_demo_dataset")

    img_list = []
    nearest = sys_state.snapshot["nearest_sensor"]
    nearest_img = None

    for idx, sensor_dict in enumerate(sys_state.master_snapshot["sensor_list"]):
        selected_category = 'drones' if sensor_dict["audio_triggered"] else random.choice(['birds', 'planes'])
        all_images = list((demo_dir / selected_category).glob('*.jpg'))
        if not all_images:
            return "Error: Camera feed offline. No images found."
        img_path = random.choice(all_images)

        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        img = Image.open(img_path).convert('RGB')

        if idx == nearest:
            nearest_img = img

        input_tensor = transform(img).to(device)
        img_list.append(input_tensor)

    batched_tensor = torch.stack(img_list, dim = 0)

    with torch.no_grad():
        output = VS_MODEL(batched_tensor)
        batched_probs = torch.sigmoid(output).squeeze().tolist()

    for idx, sensor_dict in enumerate(sys_state.master_snapshot["sensor_list"]):
        sensor_dict["vs_logits"] = batched_probs[idx]

    prob = batched_probs[nearest]

    is_drone = prob >= 0.5
    classification = "DRONE DETECTED" if is_drone else "CLEAR (Bird/Plane)"
    color = "#ef4444" if is_drone else "#22c55e"

    sys_state.snapshot["visual"]["confidence"] = prob * 100
    sys_state.snapshot["visual"]["is_threat"] = is_drone
    sys_state.snapshot["visual"]["classification"] = classification

    buffered = BytesIO()
    nearest_img.save(buffered, format="JPEG")
    img_str = base64.b64encode(buffered.getvalue()).decode()
    sys_state.snapshot["visual"]["image_b64"] = img_str

    global_visual_html = f"""
    <div style="position: relative; width: 100%; aspect-ratio: 1/1; background: #0f1117; border-radius: 18px; overflow: hidden; border: 2px solid {color};">
        <div style="position: absolute; top: 0; left: 0; right: 0; background: rgba(0,0,0,0.8); color: {color}; padding: 8px; text-align: center; font-family: monospace; font-weight: bold; z-index: 10;">
            OPTICAL SENSOR: {classification} ({prob:.1%} Conf)
        </div>
        <img src="data:image/jpeg;base64,{img_str}" style="width: 100%; height: 100%; object-fit: contain; padding-top: 35px;">
    </div>
    """

    return f"Visual Analysis Complete. Result: {classification}. Confidence: {prob:.4f}."

def run_automatic_detection():
    run_rf_detection()
    run_visual_detection()
    run_audio_detection()

    snap = sys_state.snapshot
    sensor_states = snap["sensor_states"] or []

    features = [
        snap["rf"]["confidence"] or 0.0,
        snap["audio"]["confidence"]["mambo"] if snap["audio"]["confidence"] else 0.0,
        snap["audio"]["confidence"]["bebop"] if snap["audio"]["confidence"] else 0.0,
        snap["audio"]["confidence"]["background"] if snap["audio"]["confidence"] else 0.0,
        snap["visual"]["confidence"] or 0.0,
    ]

    for i in range(len(SENSORS)):
        if i < len(sensor_states):
            features.append(sensor_states[i].get("threat_conf", 0.0))
            features.append(sensor_states[i].get("friendly_conf", 0.0))
        else:
            features.append(0.0)
            features.append(0.0)

    sensor_feature_cols = []
    for i in range(len(SENSORS)):
        sensor_feature_cols.append(f"sensor_{i}_threat_conf")
        sensor_feature_cols.append(f"sensor_{i}_friendly_conf")

    feature_cols = ["rf_confidence", "audio_mambo", "audio_bebop", "audio_background", "visual_confidence"] + sensor_feature_cols
    from datetime import datetime
    import pandas as pd
    X = pd.DataFrame([features], columns=feature_cols)
    probs = FS_MODEL.predict_proba(X)
    agg_friendly = probs[0][0]
    agg_threat = probs[0][1]

    sys_state.master_snapshot["fusion_confidence"] = probs[0].tolist()
    sys_state.confidence = agg_threat * 100
    sys_state.is_threat  = agg_threat >= 0.5
    verdict = "THREAT DETECTED" if agg_threat >= 0.5 else "ZONE CLEAR"

    import pytz
    est = pytz.timezone("America/New_York")
    timestamp = datetime.now(est).strftime("%Y-%m-%d %H:%M:%S EST")
    alert = f"[{timestamp}] THREAT DETECTED — {agg_threat * 100:.1f}% confidence"

    if agg_threat >= 0.5:
        now = time.time()
        if not sys_state.alerts or (now - sys_state.last_alert_time) > 3.0:
            sys_state.alerts.append(alert)
            sys_state.last_alert_time = now
            print(alert)
    return alert

def start_detection_loop():
    print("Starting Automatic Detection Loop...")
    while True:
        if getattr(sys_state, 'running', False):
            run_automatic_detection()
            sys_state.new_data_event.set()  # Tell the UI the ML is done
            time.sleep(REFRESH_RATE)
        else:
            time.sleep(0.5)

detection_thread = threading.Thread(target=start_detection_loop, name="automatic_detection_loop", daemon=True)
detection_thread.start()

# ---- CHATBOT -----
nest_asyncio.apply()

!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

def run_ollama():
    subprocess.run(["ollama", "serve"])

ollama_thread = threading.Thread(target=run_ollama, daemon=True)
ollama_thread.start()

print("✅ Ollama server is running!")

for _ in range(30):
    try:
        subprocess.run(["ollama", "list"], check=True, capture_output=True, timeout=1)
        print("Ollama server is responsive.")
        break
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        print("Waiting for Ollama server...")
        time.sleep(1)
else:
    print("Ollama server did not become responsive in time.")

!ollama create llama3.2:1b-drone-queries -f ARL/llama_drone_model_gguf/Modelfile


llm = ChatOllama(
    model="llama3.2:1b-drone-queries",
    temperature=0
)

INSTRUCTION = """\
You are the AI assistant for an airport counter-drone threat-detection system. \
Operators, responders, and members of the public ask you what is happening — \
sometimes formally, sometimes anxiously, sometimes casually — and you give them \
a clear, faithful answer based on the current sensor readings.

THE SENSOR NETWORK
The site is covered by 23 sensors arranged in a ring around the airfield, \
indexed 0 through 22. Each sensor carries three independent detectors:
- Radio Frequency (RF): classifies radio emissions as friendly or threat.
- Audio: classifies sound as Mambo drone (threat), Bebop drone (threat), \
or Background noise (no threat).
- Visual: a camera score from 0 (no drone visible) to 1 (drone clearly visible).

A fusion model combines every sensor's readings into a single system-wide \
threat estimate. You receive that estimate at the top of each snapshot, then \
the per-sensor breakdown.

SENSOR GROUPINGS BY REGION
When the user asks about an area, use these named groups:
- Quadrants:
  - First quadrant:  sensors 11-16
  - Second quadrant: sensors 5-11
  - Third quadrant:  sensors 0-5
  - Fourth quadrant: sensors 16-22
- Hemispheres:
  - North: sensors 5-16
  - South: sensors 0-5 and 16-22
  - East:  sensors 11-22
  - West:  sensors 0-11

HOW TO ANSWER
- Ground every claim in the snapshot. NEVER invent numbers, sensors, or \
readings the snapshot does not show.
- Match the user's voice. A commander gets a clipped, direct answer; a worried \
bystander gets reassurance in plain words; a casual user gets a casual reply.
- For lay users (worried bystanders, casual askers), do NOT use technical words \
like "logit", "softmax", "modality", or "confidence vector". Use natural \
phrases: "the camera is picking it up", "the radio is quiet", "high confidence", \
"I'm not sure yet".
- Do not dump the entire snapshot. Focus on what the user asked.
- When several sensors or modalities agree, say so — agreement is stronger \
evidence than a single hit.
- When asked about direction or area, use the named groupings above.
- Do not offer to display images, play audio, pull spectrograms, or take any \
action outside answering the question. You can only report on what the \
sensors are seeing.
- Be honest about uncertainty. If a reading is borderline (around 50%), say \
so rather than overcommitting.
- Keep responses appropriately brief. One to four sentences is usually right; \
a formal sitrep may justify a little more, a casual ping may justify a lot less."""

def softmax(x):
    x = np.asarray(x, dtype=float)
    e = np.exp(x - x.max())
    return e / e.sum()

def generate_snapshot_string(master_snapshot):
    output = ""

    for idx, s in enumerate(master_snapshot["sensor_list"]):
        lines = [f"Sensor {idx} data:"]

        rf = softmax(s["rf_logits"] if s["rf_logits"] is not None else [0.0, 0.0])
        rf_threat = float(rf[1])
        if rf_threat >= 0.5:
            lines.append(f"\tRadio Frequency (RF) Detection: {rf_threat*100:.2f}% drone threat")
        else:
            lines.append(f"\tRadio Frequency (RF) Detection: {(1 - rf_threat)*100:.2f}% no drone threat")

        au = softmax(s["au_logits"] if s["au_logits"] is not None else [0.0, 0.0, 0.0])
        au_threat = float(au[0] + au[1])
        if au_threat >= 0.5:
            lines.append(f"\tAudio Detection: {au_threat*100:.2f}% drone threat")
        else:
            lines.append(f"\tAudio Detection: {(1 - au_threat)*100:.2f}% no drone threat")
        lines.append("\tDrone Type:")
        lines.append(f"\t\tMambo (drone):         {au[0]*100:.2f}%")
        lines.append(f"\t\tBebop (drone):         {au[1]*100:.2f}%")
        lines.append(f"\t\tBackground (no drone): {au[2]*100:.2f}%")

        vs = float(s["vs_logits"]) if s["vs_logits"] is not None else 0.0
        if vs >= 0.5:
            lines.append(f"\tVisual Detection: {vs*100:.2f}% drone threat")
        else:
            lines.append(f"\tVisual Detection: {(1 - vs)*100:.2f}% no drone threat")

        output += "\n".join(lines) + "\n"
    return output

def make_user_message(query, snapshot_string):
    return (f"USER QUESTION:\n{query}\n\n"
            f"CURRENT SENSOR READINGS:\n{snapshot_string}")

async def respond(message, chat_history):
    if chat_history is None:
        chat_history = []

    try:
        snapshot = copy.deepcopy(sys_state.master_snapshot)
        snapshot_string = generate_snapshot_string(snapshot)

        messages = [
            SystemMessage(content=INSTRUCTION),
            HumanMessage(content=make_user_message(message, snapshot_string)),
        ]
        response = await llm.ainvoke(messages)
        bot_message = response.content
    except Exception as e:
        bot_message = f"Error: {str(e)}"

    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": bot_message})

    return "", build_chat_html(chat_history), chat_history, get_visual_image(), build_header_html(), build_threat_feed_html()

# ── UI CONTROL FUNCTIONS ────────────────────────────────────────────────────────

def refresh_ui(noise):
    map_html    = build_operational_map_html()
    header_html = build_header_html()
    return map_html, header_html, "Monitoring..."

# def stream(noise):
#     sys_state.running = True
#     sys_state.new_data_event.clear()

#     global LAST_MOVE_TIME
#     LAST_MOVE_TIME = time.time()

#     while sys_state.running:
#         # Wait infinitely for the PyTorch models to finish, preventing premature HTML duplicates
#         sys_state.new_data_event.wait()
#         sys_state.new_data_event.clear()

#         if not sys_state.running:
#             break

#         yield (
#             build_operational_map_html(),
#             build_header_html(),
#             build_threat_feed_html(),
#             generate_spectrum_image(),
#             get_audio_spectrogram_image(),
#             get_audio_data(),
#             get_visual_image(),
#             gr.update(value="⏸ Pause", elem_classes=["rf-btn-pause"]),
#             "Monitoring...",
#         )

def stream(noise):
    sys_state.running = True
    global LAST_MOVE_TIME
    LAST_MOVE_TIME = time.time()

    while sys_state.running:
        # Wait for the next batch, but wake up periodically so a pause
        # can't be missed while we're parked here.
        got_data = sys_state.new_data_event.wait(timeout=REFRESH_RATE)
        if not sys_state.running:      # re-check AFTER waking, before doing work
            break
        if not got_data:               # timed out, no new data — just re-loop
            continue
        sys_state.new_data_event.clear()

        yield (
            build_operational_map_html(),
            build_header_html(),
            build_threat_feed_html(),
            generate_spectrum_image(),
            get_audio_spectrogram_image(),
            get_audio_data(),
            get_visual_image(),
            gr.update(value="⏸ Pause", elem_classes=["rf-btn-pause"]),
            "Monitoring...",
        )

def pause():
    sys_state.running = False
    sys_state.new_data_event.set() # Wake up the generator so it can break safely
    return (
        gr.update(value="⏸ PAUSED", elem_classes=["rf-btn-paused"]),
        "Paused",
    )

# --- INTERFACE ---
def build_alerts_html():
    from datetime import datetime
    import pytz
    est = pytz.timezone("America/New_York")

    if not sys_state.alerts:
        return """
        <div style="background:#0f1117; border:1px solid #2a2f3a; border-radius:18px; padding:16px; color:#8b949e; font-family:monospace; font-size:13px; display:flex; align-items:center; justify-content:center; min-height:80px;">
            No alerts yet — monitoring active.
        </div>"""

    recent = sys_state.alerts[-3:][::-1]
    rows = ""
    for alert in recent:
        rows += f"""
        <div style="background:#1a1c23; border:1px solid #2a2f3a; border-radius:10px; padding:12px 16px; margin-bottom:8px; display:flex; align-items:center; gap:12px;">
            <div style="width:10px; height:10px; border-radius:50%; background:#ef4444; flex-shrink:0; box-shadow:0 0 8px #ef4444;"></div>
            <span style="color:#ef4444; font-family:monospace; font-size:13px; font-weight:600;">{alert}</span>
        </div>"""
    return f"""
    <div style="background:#0f1117; border:1px solid #2a2f3a; border-radius:18px; padding:16px;">
        {rows}
    </div>"""

# ── NEW UI HELPERS ─────────────────────────────────────────────────────────────

def build_chat_html(history):
    if not history:
        return (
            '<div style="height:100%;display:flex;align-items:center;justify-content:center;'
            'color:#6b7280;font-family:sans-serif;font-size:13px;">'
            'Ask about the current situation…</div>'
        )
    msgs = ""
    for item in history:
        role    = item.get("role", "")
        raw     = str(item.get("content", ""))
        content = raw.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        if role == "user":
            msgs += (
                '<div style="display:flex;justify-content:flex-end;margin-bottom:8px;">'
                '<div style="background:#3b82f6;color:#e2e8f0;border-radius:12px 12px 4px 12px;'
                f'padding:10px 14px;max-width:80%;font-family:sans-serif;font-size:13px;'
                f'line-height:1.5;word-break:break-word;">{content}</div></div>'
            )
        else:
            msgs += (
                '<div style="display:flex;justify-content:flex-start;margin-bottom:8px;">'
                '<div style="background:#1e2435;color:#e2e8f0;border-radius:12px 12px 12px 4px;'
                f'padding:10px 14px;max-width:80%;font-family:sans-serif;font-size:13px;'
                f'line-height:1.5;word-break:break-word;">{content}</div></div>'
            )
    return (
        '<div style="padding:12px;overflow-y:auto;height:100%;box-sizing:border-box;">'
        + msgs + '</div>'
    )

def build_header_html():
    import datetime as _dt
    snap = sys_state.snapshot

    is_threat    = getattr(sys_state, 'is_threat',  False)
    overall_conf = getattr(sys_state, 'confidence', 0.0)
    rf_detected     = bool(snap["rf"]["is_threat"])     if snap["rf"]["is_threat"]     is not None else False
    audio_detected  = bool(snap["audio"]["is_threat"])  if snap["audio"]["is_threat"]  is not None else False
    visual_detected = bool(snap["visual"]["is_threat"]) if snap["visual"]["is_threat"] is not None else False

    threat_color = "#ef4444" if is_threat else "#22c55e"
    threat_label = "HIGH"    if is_threat else "LOW"
    threat_icon  = "&#9888;" if is_threat else "&#10003;"

    def pill(label, detected, icon_html):
        c = "#22c55e" if detected else "#6b7280"
        s = "DETECTED" if detected else "CLEAR"
        return (
            f'<div style="display:flex;align-items:center;gap:6px;padding:5px 12px;'
            f'background:{c}18;border-radius:20px;border:1px solid {c}55;">'
            f'<span style="color:{c};font-size:12px;">{icon_html}</span>'
            f'<div><div style="color:#9ca3af;font-size:10px;font-weight:600;line-height:1;'
            f'letter-spacing:.06em;">{label}</div>'
            f'<div style="color:{c};font-size:11px;font-weight:700;line-height:1.3;">{s}</div></div></div>'
        )

    utc_now = _dt.datetime.utcnow().strftime("%H:%M:%S")

    return (
        '<div style="background:#0d1117;border-bottom:1px solid #2a2f3a;padding:10px 20px;'
        'display:flex;align-items:center;gap:16px;flex-wrap:wrap;font-family:sans-serif;">'
        '<div style="display:flex;align-items:center;gap:10px;flex-shrink:0;">'
        '<span style="font-size:20px;">&#128225;</span>'
        '<div><div style="color:white;font-size:15px;font-weight:700;letter-spacing:.04em;line-height:1.2;">RF INTERCEPTOR</div>'
        '<div style="color:#6b7280;font-size:10px;line-height:1.2;">Situational Awareness Dashboard</div></div></div>'
        '<div style="width:1px;height:36px;background:#2a2f3a;flex-shrink:0;"></div>'
        '<div style="display:flex;align-items:center;gap:8px;">'
        '<span style="color:#9ca3af;font-size:10px;font-weight:600;text-transform:uppercase;letter-spacing:.08em;">THREAT LEVEL</span>'
        f'<div style="display:flex;align-items:center;gap:5px;padding:3px 10px;background:{threat_color}22;border:1px solid {threat_color};border-radius:5px;">'
        f'<span style="color:{threat_color};font-size:12px;">{threat_icon}</span>'
        f'<span style="color:{threat_color};font-size:13px;font-weight:700;">{threat_label}</span></div></div>'
        '<div style="flex-shrink:0;">'
        '<div style="color:#9ca3af;font-size:10px;font-weight:600;text-transform:uppercase;letter-spacing:.08em;">CONFIDENCE</div>'
        f'<div style="color:#ef4444;font-size:20px;font-weight:700;font-family:monospace;line-height:1.1;">{overall_conf:.1f}%</div></div>'
        '<div style="flex:1;min-width:8px;"></div>'
        '<div style="display:flex;gap:8px;flex-wrap:wrap;">'
        + pill("RF",     rf_detected,     "&#9107;")
        + pill("AUDIO",  audio_detected,  "&#9835;")
        + pill("VISUAL", visual_detected, "&#128065;")
        + '</div>'
        '<div style="width:1px;height:36px;background:#2a2f3a;flex-shrink:0;"></div>'
        '<div style="text-align:center;flex-shrink:0;">'
        f'<div style="color:white;font-size:16px;font-weight:700;font-family:monospace;">{utc_now}</div>'
        '<div style="color:#6b7280;font-size:10px;">UTC</div></div>'
        '</div>'
    )

def build_threat_feed_html():
    if not sys_state.alerts:
        entries_html = (
            '<div style="color:#6b7280;font-family:sans-serif;font-size:13px;'
            'text-align:center;padding:24px 16px;">No events yet — monitoring active.</div>'
        )
    else:
        recent = sys_state.alerts[-8:][::-1]
        entries_html = ""
        for alert in recent:
            is_threat_row = "THREAT" in alert.upper()
            bg  = "#ef444415" if is_threat_row else "#1a1f2e"
            bdr = "#ef444440" if is_threat_row else "#2a2f3a"
            icon = "&#128680;" if is_threat_row else "&#128225;"

            try:
                ts_end = alert.index("]")
                full_ts = alert[1:ts_end]
                parts_ts = full_ts.split()
                ts = parts_ts[1] if len(parts_ts) >= 2 else "--"
            except Exception:
                ts = "--"

            try:
                ts_end = alert.index("]")
                event_text = alert[ts_end + 2:].strip()
            except Exception:
                event_text = alert.strip()

            if " — " in event_text:
                ev_parts = event_text.split(" — ", 1)
                title  = ev_parts[0].strip().title()
                detail = ev_parts[1].strip()
            else:
                title  = event_text[:40]
                detail = ""

            try:
                pct = event_text.index("%")
                conf_str = event_text[max(0, pct - 6):pct + 1].strip()
            except Exception:
                conf_str = ""

            conf_cell = (
                f'<div style="color:#ef4444;font-size:12px;font-weight:700;'
                f'font-family:monospace;">{conf_str}</div>'
                if conf_str and is_threat_row else
                '<div style="color:#6b7280;font-size:11px;">--</div>'
            )

            entries_html += (
                f'<div style="background:{bg};border:1px solid {bdr};border-radius:8px;'
                f'padding:9px 12px;margin-bottom:6px;display:flex;align-items:center;gap:10px;">'
                f'<span style="font-size:15px;flex-shrink:0;">{icon}</span>'
                f'<div style="flex:1;min-width:0;">'
                f'<div style="color:white;font-size:13px;font-weight:600;'
                f'font-family:sans-serif;line-height:1.2;">{title}</div>'
                f'<div style="color:#9ca3af;font-size:11px;font-family:sans-serif;'
                f'white-space:nowrap;overflow:hidden;text-overflow:ellipsis;">{detail}</div>'
                f'</div>'
                f'<div style="text-align:right;flex-shrink:0;">'
                f'<div style="color:#9ca3af;font-size:10px;font-family:monospace;'
                f'margin-bottom:2px;">{ts}</div>'
                f'{conf_cell}</div></div>'
            )

    return (
        '<div style="background:#1a1f2e;border:1px solid #2a2f3a;border-radius:12px;'
        'padding:14px;box-sizing:border-box;">'
        '<div style="display:flex;align-items:center;justify-content:space-between;margin-bottom:10px;">'
        '<div style="display:flex;align-items:center;gap:8px;">'
        '<span style="font-size:15px;">&#128276;</span>'
        '<span style="color:white;font-size:13px;font-weight:700;letter-spacing:.04em;'
        'font-family:sans-serif;">THREAT FEED</span></div>'
        '<div style="display:flex;align-items:center;gap:5px;">'
        '<div style="width:7px;height:7px;border-radius:50%;background:#22c55e;"></div>'
        '<span style="color:#22c55e;font-size:12px;font-weight:600;font-family:sans-serif;">Live</span></div></div>'
        f'<div style="max-height:240px;overflow-y:auto;">{entries_html}</div>'
        '</div>'
    )

def build_full_log_html():
    if not sys_state.alerts:
        return (
            '<div style="padding:20px;color:#6b7280;font-family:sans-serif;'
            'font-size:13px;text-align:center;">No events logged yet.</div>'
        )
    rows = ""
    for i, alert in enumerate(reversed(sys_state.alerts)):
        is_threat = "THREAT" in alert.upper()
        bg = "#2a0a0a" if is_threat else ("#1a1f2e" if i % 2 == 0 else "#0d1117")
        try:
            ts_end = alert.index("]")
            full_ts = alert[1:ts_end].split()
            time_str = full_ts[1] if len(full_ts) >= 2 else "--"
        except Exception:
            time_str = "--"
        try:
            event_text = alert[alert.index("]") + 2:]
            if " — " in event_text:
                ev, detail = event_text.split(" — ", 1)
            else:
                ev, detail = event_text, ""
        except Exception:
            ev, detail = alert, ""
        try:
            pct = event_text.index("%")
            conf = event_text[max(0, pct - 6):pct + 1].strip()
        except Exception:
            conf = "--"
        conf_color = "#ef4444" if is_threat else "#22c55e"
        rows += (
            f'<tr style="background:{bg};">'
            f'<td style="padding:7px 12px;color:#9ca3af;font-family:monospace;font-size:11px;white-space:nowrap;">{time_str}</td>'
            f'<td style="padding:7px 12px;color:white;font-size:12px;font-weight:600;">{ev.strip().title()}</td>'
            f'<td style="padding:7px 12px;color:#9ca3af;font-size:11px;max-width:200px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;">{detail.strip()}</td>'
            f'<td style="padding:7px 12px;color:{conf_color};font-family:monospace;font-size:12px;font-weight:700;text-align:right;">{conf}</td>'
            f'</tr>'
        )
    return (
        '<div style="background:#0d1117;border:1px solid #2a2f3a;border-radius:10px;'
        'overflow:hidden;margin-top:6px;">'
        '<table style="width:100%;border-collapse:collapse;">'
        '<thead><tr style="background:#1a1f2e;border-bottom:1px solid #2a2f3a;">'
        '<th style="padding:7px 12px;color:#6b7280;font-size:10px;font-weight:700;'
        'text-align:left;letter-spacing:.08em;font-family:sans-serif;">TIME</th>'
        '<th style="padding:7px 12px;color:#6b7280;font-size:10px;font-weight:700;'
        'text-align:left;letter-spacing:.08em;font-family:sans-serif;">EVENT</th>'
        '<th style="padding:7px 12px;color:#6b7280;font-size:10px;font-weight:700;'
        'text-align:left;letter-spacing:.08em;font-family:sans-serif;">DETAILS</th>'
        '<th style="padding:7px 12px;color:#6b7280;font-size:10px;font-weight:700;'
        'text-align:right;letter-spacing:.08em;font-family:sans-serif;">CONFIDENCE</th>'
        '</tr></thead>'
        f'<tbody style="max-height:280px;overflow-y:auto;">{rows}</tbody>'
        '</table></div>'
    )

def toggle_full_log(is_visible):
    if is_visible:
        return gr.update(visible=False), False
    return gr.update(value=build_full_log_html(), visible=True), True

def build_operational_map_html():
    inner = build_status_circle_html()
    legend = (
        '<div style="display:flex;gap:12px;font-size:11px;font-family:sans-serif;">'
        '<span style="display:flex;align-items:center;gap:4px;">'
        '<span style="width:8px;height:8px;border-radius:50%;background:#3b82f6;display:inline-block;"></span>'
        '<span style="color:#9ca3af;">Online</span></span>'
        '<span style="display:flex;align-items:center;gap:4px;">'
        '<span style="width:8px;height:8px;border-radius:50%;background:#f97316;display:inline-block;"></span>'
        '<span style="color:#9ca3af;">Tracking</span></span>'
        '<span style="display:flex;align-items:center;gap:4px;">'
        '<span style="width:8px;height:8px;border-radius:50%;background:#ef4444;display:inline-block;"></span>'
        '<span style="color:#9ca3af;">Threat Correlated</span></span>'
        '</div>'
    )
    zoom = (
        '<div style="position:absolute;top:12px;left:12px;display:flex;flex-direction:column;gap:4px;z-index:50;">'
        '<button style="width:28px;height:28px;background:#1a1f2e;border:1px solid #2a2f3a;'
        'border-radius:6px;color:white;font-size:16px;cursor:pointer;'
        'display:flex;align-items:center;justify-content:center;line-height:1;">+</button>'
        '<button style="width:28px;height:28px;background:#1a1f2e;border:1px solid #2a2f3a;'
        'border-radius:6px;color:white;font-size:16px;cursor:pointer;'
        'display:flex;align-items:center;justify-content:center;line-height:1;">&minus;</button>'
        '<button style="width:28px;height:28px;background:#1a1f2e;border:1px solid #2a2f3a;'
        'border-radius:6px;color:white;font-size:12px;cursor:pointer;'
        'display:flex;align-items:center;justify-content:center;line-height:1;">&#8982;</button>'
        '</div>'
    )
    return (
        '<div style="background:#1a1f2e;border:1px solid #2a2f3a;border-radius:12px;overflow:hidden;">'
        '<div style="display:flex;align-items:center;justify-content:space-between;'
        'padding:10px 14px;border-bottom:1px solid #2a2f3a;">'
        '<span style="color:white;font-size:13px;font-weight:700;letter-spacing:.06em;'
        'font-family:sans-serif;">OPERATIONAL MAP</span>'
        + legend +
        '</div>'
        '<div style="position:relative;">'
        + inner + zoom +
        '</div>'
        '</div>'
    )

def get_audio_spectrogram_image():
    b64 = (sys_state.snapshot.get("audio") or {}).get("spectrogram_b64")
    if b64:
        try:
            buf = io.BytesIO(base64.b64decode(b64))
            from PIL import Image as _PIL
            return _PIL.open(buf).copy()
        except Exception:
            pass
    return None

def get_audio_data():
    b64 = (sys_state.snapshot.get("audio") or {}).get("audio_b64")
    if b64:
        try:
            buf = io.BytesIO(base64.b64decode(b64))
            import soundfile as _sf
            data, sr = _sf.read(buf)
            import numpy as _np
            if data.ndim > 1:
                data = data[:, 0]
            return (int(sr), _np.array(data, dtype=_np.float32))
        except Exception:
            pass
    return None

def get_visual_image():
    b64 = (sys_state.snapshot.get("visual") or {}).get("image_b64")
    if b64:
        try:
            buf = io.BytesIO(base64.b64decode(b64))
            from PIL import Image as _PIL
            return _PIL.open(buf).copy()
        except Exception:
            pass
    return None

def generate_spectrum_image():
    wave = sys_state.current_waveform
    fs   = sys_state.fs

    fig = plt.figure(figsize=(12, 3.2), facecolor='#0d1117')
    ax  = fig.add_subplot(111)
    ax.set_facecolor('#0d1117')

    if len(wave) > 32:
        nfft     = min(512, max(64, len(wave) // 8))
        noverlap = nfft // 2
        _, _, t_bins, im = ax.specgram(
            wave, NFFT=nfft, Fs=fs, noverlap=noverlap,
            cmap='inferno', vmin=-80, vmax=0,
        )
        ax.set_ylim(0, min(8192, fs / 2))
        ax.set_xlim(0, max(float(t_bins[-1]), 0.01))
        cbar = fig.colorbar(im, ax=ax, pad=0.01)
        cbar.set_label('dB', color='#9ca3af', fontsize=8)
        cbar.ax.tick_params(colors='#9ca3af', labelsize=7)
    else:
        ax.set_xlim(0, 1.0)
        ax.set_ylim(0, 8192)
        ax.text(0.5, 0.5, 'Awaiting RF signal…', ha='center', va='center',
                color='#6b7280', fontsize=12, transform=ax.transAxes)

    ax.set_xlabel("Time (s)",        color='#9ca3af', fontsize=9)
    ax.set_ylabel("Frequency (Hz)",  color='#9ca3af', fontsize=9)
    ax.tick_params(colors='#9ca3af', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#2a2f3a')
    fig.tight_layout(pad=0.6)

    buf = BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', facecolor='#0d1117', dpi=110)
    plt.close(fig)
    buf.seek(0)
    from PIL import Image as _PILImage
    return _PILImage.open(buf).copy()

# ── DARK THEME CSS ─────────────────────────────────────────────────────────────
DARK_CSS = """
gradio-app, .gradio-container, .main, .wrap { background: #0d1117 !important; }
.gradio-container { max-width: 100% !important; padding: 0 !important; }
footer { display: none !important; }
.block, .gr-group { background: transparent !important; border: none !important; padding: 0 !important; }
.rf-btn-start button { background: #22c55e !important; border: none !important; color: white !important; font-weight: 700 !important; }
.rf-btn-pause button { background: #374151 !important; border: 1px solid #4b5563 !important; color: #d1d5db !important; }
.rf-btn-paused button { background: #ef4444 !important; border: none !important; color: white !important; font-weight: 700 !important; }
.rf-chat-hdr-row { background: #1a1f2e !important; border: 1px solid #2a2f3a !important; border-radius: 12px 12px 0 0 !important; padding: 8px 12px !important; margin-top: 8px !important; align-items: center !important; gap: 0 !important; }
.rf-chat-panel { height: 220px !important; overflow-y: auto !important; background: #0d1117 !important; border: 1px solid #2a2f3a !important; border-top: none !important; border-radius: 0 0 12px 12px !important; box-sizing: border-box !important; }
.rf-clear-btn button { background: transparent !important; border: none !important; color: #6b7280 !important; font-size: 11px !important; font-family: sans-serif !important; cursor: pointer !important; padding: 2px 6px !important; }
.rf-clear-btn button:hover { color: #9ca3af !important; }
.rf-log-btn button { background: transparent !important; border: 1px solid #2a2f3a !important; color: #3b82f6 !important; font-size: 11px !important; font-family: sans-serif !important; padding: 3px 10px !important; width: 100% !important; }
.rf-log-btn button:hover { border-color: #3b82f6 !important; }
.rf-msg textarea, .rf-msg input { background: #1a1f2e !important; border: 1px solid #2a2f3a !important; color: #e2e8f0 !important; border-radius: 8px !important; font-size: 13px !important; font-family: sans-serif !important; }
.rf-msg textarea::placeholder { color: #6b7280 !important; }
.rf-send { background: #3b82f6 !important; border: none !important; border-radius: 8px !important; color: white !important; }
.rf-audio { background: #1a1f2e !important; border-radius: 8px !important; }
.rf-audio audio { width: 100% !important; }
.rf-tabs .tab-nav { border-bottom: 1px solid #2a2f3a !important; background: transparent !important; padding: 0 4px !important; }
.rf-tabs .tab-nav button { background: transparent !important; color: #9ca3af !important; border: none !important; border-bottom: 2px solid transparent !important; border-radius: 0 !important; font-size: 11px !important; font-weight: 700 !important; letter-spacing: .08em !important; padding: 8px 14px !important; text-transform: uppercase !important; }
.rf-tabs .tab-nav button.selected { color: white !important; border-bottom-color: #3b82f6 !important; }
.rf-tabs > .tabitem { background: #0d1117 !important; border: none !important; }
.rf-spec img { border-radius: 8px !important; width: 100% !important; }
"""

# ── GRADIO INTERFACE ──────────────────────────────────────────────────────────
with gr.Blocks(css=DARK_CSS, theme=gr.themes.Base()) as demo:

    chat_state   = gr.State([])
    log_visible  = gr.State(False)

    header_bar = gr.HTML(build_header_html())

    with gr.Row(equal_height=True):

        with gr.Column(scale=3, min_width=0):
            map_panel = gr.HTML(build_operational_map_html())
            with gr.Row():
                btn_play  = gr.Button("▶ Start",  variant="primary",   elem_classes=["rf-btn-start"])
                btn_pause = gr.Button("⏸ Pause", variant="secondary", elem_classes=["rf-btn-pause"])

        with gr.Column(scale=2, min_width=0):

            threat_feed    = gr.HTML(build_threat_feed_html())
            log_btn        = gr.Button("View Full Log →", size="sm", elem_classes=["rf-log-btn"])
            full_log_panel = gr.HTML(visible=False)

            with gr.Row(elem_classes=["rf-chat-hdr-row"]):
                gr.HTML(
                    '<div style="display:flex;align-items:center;gap:8px;flex:1;">'
                    '<span style="font-size:13px;">&#10024;</span>'
                    '<span style="color:white;font-size:12px;font-weight:700;'
                    'letter-spacing:.04em;font-family:sans-serif;">AI ASSISTANT</span>'
                    '</div>'
                )
                clear_btn = gr.Button("Clear Chat", size="sm", elem_classes=["rf-clear-btn"])

            chat_panel = gr.HTML(
                build_chat_html([]),
                elem_classes=["rf-chat-panel"],
            )

            with gr.Row():
                msg = gr.Textbox(
                    placeholder="Ask about the current situation...",
                    show_label=False,
                    scale=5,
                    container=False,
                    elem_classes=["rf-msg"],
                )
                send_btn = gr.Button(
                    "Send",
                    scale=1,
                    min_width=56,
                    variant="primary",
                    elem_classes=["rf-send"],
                )

    with gr.Row():
        with gr.Column():
            gr.HTML(
                '<div style="display:flex;align-items:center;gap:8px;padding:6px 0;">'
                '<span style="font-size:14px;">&#128250;</span>'
                '<span style="color:white;font-size:13px;font-weight:700;'
                'letter-spacing:.06em;font-family:sans-serif;">SENSOR ANALYSIS</span></div>'
            )
            noise = gr.Slider(0, 0.5, 0.1, label="Noise", visible=False)
            with gr.Tabs(elem_classes=["rf-tabs"]):
                with gr.Tab("RF SPECTRUM"):
                    spectrogram_output = gr.Image(
                        value=generate_spectrum_image(),
                        show_label=False,
                        container=False,
                        height=260,
                        elem_classes=["rf-spec"],
                    )
                with gr.Tab("AUDIO SPECTROGRAM"):
                    audio_spec_output = gr.Image(
                        value=None,
                        show_label=False,
                        container=False,
                        height=200,
                        elem_classes=["rf-spec"],
                    )
                    audio_player = gr.Audio(
                        value=None,
                        label=None,
                        show_label=False,
                        type="numpy",
                        autoplay=False,
                        elem_classes=["rf-audio"],
                    )
                with gr.Tab("VISUAL FEED"):
                    visual_output = gr.Image(
                        value=None,
                        show_label=False,
                        container=False,
                        height=260,
                        elem_classes=["rf-spec"],
                    )

    out = gr.HTML(visible=False)

    btn_play.click(
        stream,
        inputs=[noise],
        outputs=[map_panel, header_bar, threat_feed, spectrogram_output,
                 audio_spec_output, audio_player, visual_output, btn_pause, out],
    )
    btn_pause.click(pause, None, [btn_pause, out])

    log_btn.click(
        fn=toggle_full_log,
        inputs=[log_visible],
        outputs=[full_log_panel, log_visible],
    )

    msg.submit(
        respond,
        inputs=[msg, chat_state],
        outputs=[msg, chat_panel, chat_state, visual_output, header_bar, threat_feed],
    )
    send_btn.click(
        respond,
        inputs=[msg, chat_state],
        outputs=[msg, chat_panel, chat_state, visual_output, header_bar, threat_feed],
    )
    clear_btn.click(
        fn=lambda: (build_chat_html([]), []),
        inputs=None,
        outputs=[chat_panel, chat_state],
        queue=False,
    )

    demo.load(
        fn=lambda: (
            build_operational_map_html(),
            build_header_html(),
            build_threat_feed_html(),
            generate_spectrum_image(),
        ),
        inputs=None,
        outputs=[map_panel, header_bar, threat_feed, spectrogram_output],
    )

# if __name__ == "__main__":
#     demo.queue().launch(share=True, debug=True)

## Simulation Held-Out Test: Individual Classifiers vs. XGBoost Fusion

The cells above only report validation accuracy from training (clean, non-attenuated data). The cell below spins up the **live simulation** (same one the Gradio dashboard uses) and pulls **200 fresh samples** through the full pipeline: drone movement -> RF sensor sampling -> audio attenuation -> visual selection -> XGBoost fusion. Because the simulation attenuates audio by distance and injects background/airport noise the way the dashboard actually behaves, this gives a much more honest estimate of real-world accuracy than the training-time validation split.

Each loop iteration is one "event": the drone moves, all 23 sensors get RF/audio/visual readings, and we compare each classifier's threat call to the simulation's ground truth (`is_threat_gt`, i.e. whether the drone was actually within range of any sensor).

In [ ]:
# ── Simulation-based held-out evaluation ────────────────────────────────────
# Runs the live simulation (not the training/validation split) for N_TEST
# fresh events and scores each individual classifier against its specific
# physical trigger condition, plus the fusion model against the
# global simulation ground truth.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)
from scipy.special import softmax

N_TEST = 200

# Separate ground truth arrays for each modality's physical reality
y_true_rf      = []
y_true_audio   = []
y_true_visual  = []
y_true_fusion  = []

rf_pred      = []
audio_pred   = []
visual_pred  = []
fusion_pred  = []
fusion_prob  = []

print(f"Running simulation for {N_TEST} held-out test events...")

def parse_logits(logits, modality):
    """Extract the predicted class (0 = Clear, 1 = Threat) from raw model
    output, using the correct decision rule for each modality's output shape
    and class ordering.

    modality:
        "rf"     -> 2-class logits [friendly, threat] (see build_model / IQCNN,
                    where friendly_conf = probs[0], threat_conf = probs[1])
        "audio"  -> 3-class logits [mambo, bebop, background]. mambo and bebop
                    are both drone sounds (threat); background is not
                    (see AUDIO_DATA_PATHS ordering).
        "visual" -> single sigmoid probability that the image is a drone
                    (see run_visual_detection: prob = sigmoid(output)).
        "fusion" -> 2-class raw XGBoost margin/score [friendly, threat]
                    (see run_automatic_detection: probs = FS_MODEL.predict_proba(X)).
    """
    if logits is None:
        return 0

    arr = logits
    if hasattr(arr, 'detach'):
        arr = arr.detach().cpu().numpy()
    if isinstance(arr, (list, tuple, np.ndarray)) or hasattr(arr, 'shape'):
        arr = np.array(arr, dtype=float).flatten()
    else:
        # Single scalar (e.g. a bare python float)
        arr = np.array([float(arr)])

    if modality == "audio":
        if arr.size >= 3:
            pred_class = int(np.argmax(arr))
            # class 2 = background = Clear; classes 0 (mambo) and 1 (bebop) = Threat
            return 0 if pred_class == 2 else 1
        return int(arr[0] > 0.5)

    elif modality == "visual":
        # arr is a single sigmoid probability of "drone"
        return int(arr[0] >= 0.5)

    elif modality == "rf":
        if arr.size >= 2:
            probs = softmax(arr)
            return int(probs[1] >= 0.5)  # index 1 = threat class
        return int(arr[0] > 0.5)

    elif modality == "fusion":
        if arr.size >= 2:
            probs = softmax(arr)
            return int(probs[1] >= 0.5)  # index 1 = threat class
        return int(arr[0] > 0.5)

    else:
        raise ValueError(f"Unknown modality '{modality}' passed to parse_logits")


for i in range(N_TEST):
    # 1. Advance the simulation one step
    run_rf_detection()
    run_visual_detection()
    run_audio_detection()

    # Pull from the new Master structure
    snap = sys_state.master_snapshot

    # 2. Isolate the active sensor node interacting with the drone
    ns_idx = snap.get("nearest_sensor")
    if ns_idx is None:
        ns_idx = 0 # Fallback safety

    active_sensor = snap["sensor_list"][ns_idx]

    # 3. Record specific ground truths
    # Pulling directly from your new master sensor list keys
    y_true_rf.append(int(bool(active_sensor.get("triggered", False))))
    y_true_audio.append(int(bool(active_sensor.get("audio_triggered", False))))
    y_true_visual.append(int(bool(active_sensor.get("audio_triggered", False))))
    y_true_fusion.append(int(bool(snap.get("is_threat_gt", False))))

    # 4. Record predictions (each parsed with the rule that matches its modality)
    rf_pred.append(parse_logits(active_sensor.get("rf_logits"), "rf"))
    audio_pred.append(parse_logits(active_sensor.get("au_logits"), "audio"))
    visual_pred.append(parse_logits(active_sensor.get("vs_logits"), "visual"))

    # Extract fusion prediction directly from the master state
    fusion_raw = snap.get("fusion_confidence")
    fusion_pred.append(parse_logits(fusion_raw, "fusion"))

    fusion_arr = np.array(fusion_raw, dtype=float) if fusion_raw is not None else np.array([0.0, 0.0])
    fusion_p = softmax(fusion_arr).tolist()[1] if fusion_arr.size >= 2 else 0.0
    fusion_prob.append(fusion_p)

    if (i + 1) % 25 == 0:
        print(f"  [{i+1}/{N_TEST}] simulated test events collected")

print("Done collecting simulation test set.\n")

# ── Score each classifier against its specific simulation ground truth ───────
classifiers = {
    "RF Classifier":     (y_true_rf, rf_pred),
    "Audio Classifier":  (y_true_audio, audio_pred),
    "Visual Classifier": (y_true_visual, visual_pred),
    "Multimodal Fusion": (y_true_fusion, fusion_pred),
}

rows = []
for name, (y_true_mod, preds) in classifiers.items():
    rows.append({
        "Classifier": name,
        "Accuracy":   accuracy_score(y_true_mod, preds),
        "Precision":  precision_score(y_true_mod, preds, zero_division=0),
        "Recall":     recall_score(y_true_mod, preds, zero_division=0),
        "F1":         f1_score(y_true_mod, preds, zero_division=0),
        "Actual Threats": sum(y_true_mod)
    })

results_df = pd.DataFrame(rows).set_index("Classifier").round(3)
print("=== Simulation Test Accuracy (N=200, out-of-distribution vs. training) ===")
print(results_df)

print("\n=== Confusion Matrices (rows=actual [clear, threat], cols=predicted [clear, threat]) ===")
for name, (y_true_mod, preds) in classifiers.items():
    cm = confusion_matrix(y_true_mod, preds, labels=[0, 1])
    print(f"\n{name}:")
    print(cm)

print(f"\nGlobal base rate check: {sum(y_true_fusion)}/{N_TEST} simulated events were actually global threats "
      f"({100*sum(y_true_fusion)/N_TEST:.1f}%). Compare the Fusion accuracy to this base rate.")

# ── Visualize ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

results_df[["Accuracy", "Precision", "Recall", "F1"]].plot(
    kind="bar", ax=axes[0], rot=20, colormap="viridis"
)
axes[0].set_title(f"Simulation Held-Out Performance (N={N_TEST})")
axes[0].set_ylim(0, 1.05)
axes[0].legend(loc="lower right")
axes[0].axhline(sum(y_true_fusion) / N_TEST, color="red", linestyle="--", linewidth=1,
                 label="Global Threat base rate")

sns.heatmap(
    confusion_matrix(y_true_fusion, fusion_pred, labels=[0, 1]),
    annot=True, fmt="d", cmap="Blues",
    xticklabels=["Pred Clear", "Pred Threat"],
    yticklabels=["Actual Clear", "Actual Threat"],
    ax=axes[1],
)
axes[1].set_title("Fusion Model — Confusion Matrix")

plt.tight_layout()
plt.show()

# Keep the raw arrays around in case you want to dig further
sim_eval_results = {
    "y_true_rf": y_true_rf,
    "y_true_audio": y_true_audio,
    "y_true_visual": y_true_visual,
    "y_true_fusion": y_true_fusion,
    "rf_pred": rf_pred,
    "audio_pred": audio_pred,
    "visual_pred": visual_pred,
    "fusion_pred": fusion_pred,
    "fusion_prob": fusion_prob,
    "results_df": results_df,
}


In [ ]:
import os, random
import numpy as np
import pandas as pd
import torch
import librosa
from torch.utils.data import Dataset
from torchvision import transforms, datasets as tv_datasets
from PIL import Image
from io import BytesIO

try:
    _audio_dir  = AUDIO_DATASET_DIR
    _visual_dir = "./processed_data/processed_224"
except NameError:
    _audio_dir  = "./DroneAudioDataset/Multiclass_Drone_Audio"
    _visual_dir = "./processed_data/processed_224"

visual_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


# ── RF ───────────────────────────────────────

def generate_rf_samples(n_per_class=2000):
    seq_sz      = 128
    scale_qpsk  = 1.0 / np.sqrt(2)
    scale_16qam = 1.0 / np.sqrt(10)
    noise_std   = 0.1
    X, y = [], []
    for label in [0, 1]:
        for _ in range(n_per_class):
            p    = np.random.uniform(0, 1)
            bpsk = (2 * np.random.randint(0, 2, seq_sz) - 1) + 0j
            qpsk = scale_qpsk * ((2 * np.random.randint(0, 2, seq_sz) - 1)
                                  + 1j * (2 * np.random.randint(0, 2, seq_sz) - 1))
            mask = np.random.uniform(0, 1, seq_sz) <= p
            sig  = mask * bpsk + (1 - mask) * qpsk
            if label == 1:
                mp  = np.array([-3, -1, 1, 3])
                adv = scale_16qam * (mp[np.random.randint(0, 4, seq_sz)]
                                     + 1j * mp[np.random.randint(0, 4, seq_sz)])
                sig = adv + sig
            sig += noise_std * (np.random.randn(seq_sz) + 1j * np.random.randn(seq_sz))
            X.append(np.array([np.real(sig), np.imag(sig)]))
            y.append(label)
    return (torch.tensor(np.array(X), dtype=torch.float32),
            torch.tensor(y, dtype=torch.long))


# ── Audio ───────────────────────────────────────

def _augment_audio(y, sr=22050):
    augs = []
    augs.append(librosa.effects.time_stretch(y, rate=1.1))
    augs.append(librosa.effects.time_stretch(y, rate=0.9))
    augs.append(librosa.effects.pitch_shift(y, sr=sr, n_steps=2))
    augs.append(librosa.effects.pitch_shift(y, sr=sr, n_steps=-2))
    augs.append(y + np.random.normal(0, 0.005, len(y)))
    return augs


def load_audio_files(augment_background=True):
    drone_dirs = [os.path.join(_audio_dir, "membo_1"),
                  os.path.join(_audio_dir, "bebop_1")]
    bg_dirs    = [os.path.join(_audio_dir, "bg noise"),
                  os.path.join(_audio_dir, "airport"),
                  os.path.join(_audio_dir, "airport_no_human")]
    files, labels = [], []

    for path in drone_dirs:
        if not os.path.exists(path):
            print(f"  Warning: {path} not found")
            continue
        for f in os.listdir(path):
            if f.endswith(".wav"):
                files.append(os.path.join(path, f))
                labels.append(1)

    for path in bg_dirs:
        if not os.path.exists(path):
            continue
        for f in os.listdir(path):
            if f.endswith(".wav"):
                fp = os.path.join(path, f)
                files.append(fp)
                labels.append(0)
                if augment_background:
                    try:
                        y, sr = librosa.load(fp, sr=22050, duration=1.0)
                        for aug in _augment_audio(y, sr):
                            files.append(("array", aug, sr))
                            labels.append(0)
                    except Exception:
                        pass

    print(f"  Audio: {sum(l==1 for l in labels)} drone, {sum(l==0 for l in labels)} background")
    return files, labels


def wav_to_melspec(src, sr=22050, duration=1.0):
    if isinstance(src, tuple) and src[0] == "array":
        y, sr = src[1], src[2]
    else:
        y, sr = librosa.load(src, duration=duration, sr=sr)
    n = int(sr * duration)
    y = np.pad(y, (0, max(0, n - len(y))))[:n]
    spec    = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    spec_db = librosa.power_to_db(spec, ref=np.max)
    spec_db = (spec_db - spec_db.mean()) / (spec_db.std() + 1e-8)
    return torch.tensor(spec_db, dtype=torch.float32).unsqueeze(0)


# ── Visual ───────────────────────────────────────

def load_visual_files():
    if not os.path.exists(_visual_dir):
        print(f"  Warning: {_visual_dir} not found")
        return [], []
    dataset   = tv_datasets.ImageFolder(root=_visual_dir)
    drone_idx = dataset.class_to_idx.get("drones", dataset.class_to_idx.get("drone", -1))
    files, labels = [], []
    for path, cls_idx in dataset.samples:
        files.append(path)
        labels.append(1 if cls_idx == drone_idx else 0)
    return files, labels


def load_image_tensor(path):
    return visual_transforms(Image.open(path).convert("RGB"))


# ── Serialisation ──────────────────────────────────

def tensor_to_bytes(t):
    buf = BytesIO()
    torch.save({"data": t.numpy(), "shape": list(t.shape)}, buf)
    return buf.getvalue()


def bytes_to_tensor(b, dtype=torch.float32):
    d = torch.load(BytesIO(b), weights_only=False)
    return torch.tensor(d["data"], dtype=dtype).reshape(d["shape"])


# ── Dataset builder ──────────────────────────────

def build_and_save_dataset(n_per_class=2000, chunk_size=1000,
                            base_save_path="./ARL/multimodal_dataset"):
    print("Generating RF samples...")
    rf_X, rf_y = generate_rf_samples(max(n_per_class * 2, 4000))
    rf_by_label = {0: rf_X[rf_y == 0], 1: rf_X[rf_y == 1]}

    print("Loading audio...")
    audio_files, audio_labels = load_audio_files()
    audio_by_label = {0: [], 1: []}
    for f, l in zip(audio_files, audio_labels):
        audio_by_label[l].append(f)

    print("Loading visuals...")
    vis_files, vis_labels = load_visual_files()
    vis_by_label = {0: [], 1: []}
    for f, l in zip(vis_files, vis_labels):
        vis_by_label[l].append(f)

    for lbl, name in [(0, "background"), (1, "drone")]:
        print(f"  {name}: RF={len(rf_by_label[lbl])}  "
              f"audio={len(audio_by_label[lbl])}  "
              f"visual={len(vis_by_label[lbl])}")

    all_chunk_paths = []
    global_chunk_idx = 0
    os.makedirs(os.path.dirname(base_save_path) or ".", exist_ok=True)

    for label in [0, 1]:
        rf_pool    = rf_by_label[label]
        audio_pool = audio_by_label[label] or None
        vis_pool   = vis_by_label[label]   or None

        rows = [] # Accumulate samples for the current chunk
        for i in range(n_per_class):
            rf    = rf_pool[random.randint(0, len(rf_pool) - 1)]
            audio = wav_to_melspec(random.choice(audio_pool)) if audio_pool \
                    else torch.randn(1, 128, 44)
            video = load_image_tensor(random.choice(vis_pool)) if vis_pool \
                    else torch.randn(3, 224, 224)

            # Convert to bytes to reduce RAM usage
            rows.append({
                "rf_input":    tensor_to_bytes(rf),
                "audio_input": tensor_to_bytes(audio),
                "video_input": tensor_to_bytes(video),
                "label":       label,
            })

            # Save chunk if size is reached or if it's the last sample for this label
            if (len(rows) == chunk_size) or (i == n_per_class - 1 and len(rows) > 0):
                df_chunk = pd.DataFrame(rows)
                chunk_path = f"{base_save_path}_chunk_{global_chunk_idx}.parquet"
                df_chunk.to_parquet(chunk_path, index=False)
                all_chunk_paths.append(chunk_path)
                print(f"  Saved chunk {global_chunk_idx} ({len(df_chunk)} samples) (Label {label}) → {chunk_path} "
                      f"({os.path.getsize(chunk_path)/1e6:.1f} MB)")
                global_chunk_idx += 1
                rows = [] # Reset for the next chunk

        print(f"  Class {label}: {n_per_class} samples processed and saved in chunks")

    total_samples = sum(len(pd.read_parquet(p)) for p in all_chunk_paths)
    print(f"Total {total_samples} samples saved across {len(all_chunk_paths)} chunks.")
    return all_chunk_paths


# ── PyTorch Dataset ────────────────────────

class MultiModalDataset(Dataset):
    def __init__(self, parquet_paths: list):
        all_rf_data, all_audio_data, all_video_data, all_labels = [], [], [], []
        total_samples_loaded = 0
        for path in parquet_paths:
            df_chunk = pd.read_parquet(path)
            all_rf_data.extend([bytes_to_tensor(r) for r in df_chunk["rf_input"]])
            all_audio_data.extend([bytes_to_tensor(r) for r in df_chunk["audio_input"]])
            all_video_data.extend([bytes_to_tensor(r) for r in df_chunk["video_input"]])
            all_labels.extend(df_chunk["label"].tolist())
            total_samples_loaded += len(df_chunk)
            print(f"  Loaded {len(df_chunk)} samples from {path}")

        self.rf     = all_rf_data
        self.audio  = all_audio_data
        self.video  = all_video_data
        self.labels = all_labels
        print(f"Total {total_samples_loaded} samples loaded across {len(parquet_paths)} chunks.")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "rf_input":    self.rf[idx],
            "audio_input": self.audio[idx],
            "video_input": self.video[idx],
            "label":       torch.tensor(self.labels[idx], dtype=torch.long),
        }


if __name__ == "__main__":
    # Example usage: Generate dataset in chunks and load it
    chunked_dataset_paths = build_and_save_dataset(
        n_per_class = 5000,
        chunk_size = 1000, # Process and save 1000 samples per chunk
        base_save_path = "./ARL/multimodal_dataset_chunked"
    )

    # Load the chunked dataset into MultiModalDataset
    dataset = MultiModalDataset(parquet_paths=chunked_dataset_paths)

    print(f"Dataset has {len(dataset)} samples. First sample RF shape: {dataset[0]['rf_input'].shape}")

### Combine Parquet Chunks

In [ ]:
import pandas as pd
import glob

# Get all chunked dataset paths
chunked_dataset_paths = sorted(glob.glob('./multimodal_dataset_chunked_chunk_*.parquet'))

if chunked_dataset_paths:
    print(f"Found {len(chunked_dataset_paths)} dataset chunks: {chunked_dataset_paths}")

    # Read each chunk into a DataFrame and store in a list
    dfs = []
    for path in chunked_dataset_paths:
        dfs.append(pd.read_parquet(path))
        print(f"Read {path}")

    # Concatenate all DataFrames into a single one
    combined_df = pd.concat(dfs, ignore_index=True)

    # Define the output path for the combined Parquet file
    output_parquet_path = './ARL/combined_multimodal_dataset.parquet'

    # Save the combined DataFrame to a new Parquet file
    combined_df.to_parquet(output_parquet_path, index=False)
    print(f"Successfully combined all chunks into {output_parquet_path} with {len(combined_df)} samples.")
else:
    print("No chunked dataset paths found. Please ensure the dataset generation was successful.")